In [1]:
!pip install pyiqa --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.2/276.2 kB 19.0 MB/s eta 0:00:00


In [2]:
# Install and authorize gdown for folders
#!pip install gdown --upgrade
#!gdown --folder "https://drive.google.com/drive/folders/1eN8Mn1wruhhu98wWJmaQOmYvQfbR0Z20" -O /kaggle/working

#!unzip -q -o /kaggle/working/ST/IC17/test/IC17_test.zip -d /kaggle/working/stt_test
#!unzip -q -o /kaggle/working/ST/IC17/train/IC17_train.zip -d /kaggle/working/stt_train
#!unzip -q /kaggle/working/HT/ht_test.zip -d /kaggle/working/htt_test
#!unzip -q /kaggle/working/HT/ht_train.zip -d /kaggle/working/htt_train

In [ ]:
# ============================================================================
# Model.py
# ============================================================================

import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange


# ============================================================================
# CA-SSM (Content-Adaptive State-Space Model)
# ============================================================================

def _linear_scan(a, b):
    """Inclusive scan of the first-order linear recurrence

        h_t = a_t * h_{t-1} + b_t,   h_0 = 0,   t = 1..L   (along dim 1)

    for tensors a, b of shape (N, L, D). The pair (a_t, b_t) composes with the
    associative operator (a1, b1) o (a2, b2) = (a1 * a2, a2 * b1 + b2), so the
    prefixes are computed with a work-efficient odd/even parallel scan:
    O(L) work and O(log L) depth, with no custom CUDA kernel.
    """
    L = a.shape[1]
    if L == 1:
        return b
    if L % 2 == 1:  # pad with the identity element (a = 1, b = 0)
        a = F.pad(a, (0, 0, 0, 1), value=1.0)
        b = F.pad(b, (0, 0, 0, 1), value=0.0)
    a_even, a_odd = a[:, 0::2], a[:, 1::2]
    b_even, b_odd = b[:, 0::2], b[:, 1::2]
    # Combine neighbouring pairs (2i, 2i+1) and solve the half-length problem:
    # its solution is the state at every odd position.
    h_odd = _linear_scan(a_even * a_odd, a_odd * b_even + b_odd)
    # State at even positions: h_{2i} = a_{2i} * h_{2i-1} + b_{2i}.
    h_prev = F.pad(h_odd[:, :-1], (0, 0, 1, 0))
    h_even = a_even * h_prev + b_even
    h = torch.stack((h_even, h_odd), dim=2).reshape(a.shape)
    return h[:, :L]


class _SelectiveScan(torch.autograd.Function):
    """Linear recurrence with an adjoint backward pass.

    Forward : h_t = a_t * h_{t-1} + b_t
    Backward: lam_t = dL/dh_t + a_{t+1} * lam_{t+1}   (same scan, reverse time)
              dL/db_t = lam_t,   dL/da_t = lam_t * h_{t-1}
    Only a and h are stored, so memory stays linear in the sequence length.
    """

    @staticmethod
    def forward(ctx, a, b):
        with torch.no_grad():
            h = _linear_scan(a, b)
        ctx.save_for_backward(a, h)
        return h

    @staticmethod
    def backward(ctx, grad_h):
        a, h = ctx.saved_tensors
        a_next = F.pad(a[:, 1:], (0, 0, 0, 1), value=0.0)          # a_{t+1}
        lam = _linear_scan(a_next.flip(1), grad_h.flip(1)).flip(1)
        h_prev = F.pad(h[:, :-1], (0, 0, 1, 0))                    # h_{t-1}
        return lam * h_prev, lam


def selective_scan(a, b):
    return _SelectiveScan.apply(a, b)


class CA_SSM(nn.Module):
    """
    CA-SSM: Content-Adaptive State-Space Model.

    A bidirectional selective state-space recurrence run along the reading
    axis (each row of the feature map is one sequence):

        g_t    = sigmoid(W_g2 GELU(W_g1 u_t))                  content gate
        Delta_t = g_t * softplus(W_dt u_t + b_dt)              gated step
        B_t = W_B u_t,  C_t = W_C u_t,  A^(d) = -exp(A_log^(d))
        Abar_t = exp(Delta_t * A^(d))                          ZOH
        Bbar_t = (1 - Abar_t) / (-A^(d)) * B_t                 ZOH
        h_t^(->) = Abar_t * h_{t-1}^(->) + Bbar_t * u_t        left to right
        h_t^(<-) = Abar_t * h_{t+1}^(<-) + Bbar_t * u_t        right to left
        o_t^(d)  = C_t * h_t^(d) + D^(d) * u_t

    When g_t -> 0, Delta_t -> 0, so Abar_t -> 1 and Bbar_t -> 0: the token
    leaves the state unchanged, and the context gathered on each side of it is
    carried across. The state is diagonal (one scalar per channel and per
    direction), i.e. its dimension is d_inner for each direction.
    """

    def __init__(self, dim, expand=2, d_conv=3,
                 dt_min=1e-3, dt_max=1e-1, a_min=1.0, a_max=16.0):
        super().__init__()
        self.dim = dim
        self.d_inner = int(dim * expand)
        d = self.d_inner

        # Input projection into a content stream u and a modulation stream r
        self.in_proj = nn.Linear(dim, 2 * d)
        # Local 2-D context before the row-wise scans
        self.dwconv = nn.Conv2d(d, d, d_conv, padding=d_conv // 2, groups=d)

        # Content gate g_t (scales the discretisation step)
        self.content_gate = nn.Sequential(
            nn.Linear(d, d // 4),
            nn.GELU(),
            nn.Linear(d // 4, d),
            nn.Sigmoid(),
        )
        # Input-dependent step, input matrix B_t and output matrix C_t
        self.dt_proj = nn.Linear(d, d)
        self.x_proj = nn.Linear(d, d)
        self.c_proj = nn.Linear(d, d)

        # Direction-specific decay rates and skip terms: index 0 = forward (->),
        # index 1 = backward (<-). -A is initialised on [a_min, a_max] so that the
        # channels start with memory horizons ranging from a few to ~100 columns.
        a_init = torch.linspace(a_min, a_max, d).log()
        self.A_log = nn.Parameter(torch.stack((a_init, a_init.clone())))
        self.D = nn.Parameter(torch.ones(2, d))

        # Output projection of the concatenated directions
        self.out_proj = nn.Linear(2 * d, dim)

        # Step-size bias initialised so that softplus(bias) ~ U[dt_min, dt_max]
        # on a log scale (inverse softplus).
        dt = torch.exp(torch.rand(d) * (math.log(dt_max) - math.log(dt_min))
                       + math.log(dt_min))
        with torch.no_grad():
            self.dt_proj.bias.copy_(dt + torch.log(-torch.expm1(-dt)))

    def _direction(self, u, delta, Bt, Ct, k, reverse):
        """One scan direction. u, delta, Bt, Ct: (N, W, d_inner)."""
        # The scan runs in at least float32, also under mixed precision
        dtype = torch.promote_types(u.dtype, torch.float32)
        A = -torch.exp(self.A_log[k].to(dtype))                # A < 0
        dA = delta.to(dtype) * A
        a = torch.exp(dA)                                      # Abar_t in (0, 1]
        b = (-torch.expm1(dA) / (-A)) * Bt.to(dtype) * u.to(dtype)  # Bbar_t * u_t
        if reverse:
            a, b = a.flip(1), b.flip(1)
        h = selective_scan(a, b)
        if reverse:
            h = h.flip(1)
        return Ct * h.to(u.dtype) + self.D[k] * u

    def forward(self, x):
        Bsz, C, H, W = x.shape
        residual = x

        u, r = self.in_proj(rearrange(x, 'b c h w -> b h w c')).chunk(2, dim=-1)
        u = F.silu(self.dwconv(rearrange(u, 'b h w d -> b d h w')))

        # Each row is one sequence along the reading axis (state reset per row)
        u = rearrange(u, 'b d h w -> (b h) w d')
        r = rearrange(r, 'b h w d -> (b h) w d')

        g = self.content_gate(u)
        delta = g * F.softplus(self.dt_proj(u))
        Bt = self.x_proj(u)
        Ct = self.c_proj(u)

        o_fwd = self._direction(u, delta, Bt, Ct, 0, reverse=False)
        o_bwd = self._direction(u, delta, Bt, Ct, 1, reverse=True)

        o = torch.cat((o_fwd, o_bwd), dim=-1) * F.silu(torch.cat((r, r), dim=-1))
        y = self.out_proj(o)
        y = rearrange(y, '(b h) w c -> b c h w', b=Bsz, h=H)
        return y + residual


# ============================================================================
# CG-CSWA (Character-Guided Cyclic Shift Window Attention)
# ============================================================================

class CG_CSWA(nn.Module):
    """
    CG-CSWA: Character-Guided Cyclic Shift Window Attention

    Novel Contributions:
    1. Predicts character boundaries to guide attention
    2. Boundary-aware attention bias prevents bleeding across characters
    3. Proper cyclic shifting with attention masking
    4. Text-specific attention mechanism

    This addresses the limitation of generic attention that doesn't understand
    character structure, leading to artifacts in degraded text restoration.
    """
    def __init__(self, dim, num_heads=8, window_size=8, shift_size=0):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

        # Character guidance: boundary map B(x) and boundary-aware attention bias.
        self.boundary_logits = None  # kept for the auxiliary boundary loss
        self.boundary_predictor = nn.Sequential(
            nn.Conv2d(dim, dim // 2, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(dim // 2, dim // 4, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(dim // 4, 1, 1)  # logits of the character-boundary map B(x)
        )
        self.boundary_bias_weight = nn.Parameter(torch.zeros(1, num_heads, 1, 1))

        # Standard relative position bias
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size - 1) * (2 * window_size - 1), num_heads)
        )

        # Get pair-wise relative position index
        coords_h = torch.arange(window_size)
        coords_w = torch.arange(window_size)
        coords = torch.stack(torch.meshgrid([coords_h, coords_w], indexing='ij'))
        coords_flatten = torch.flatten(coords, 1)
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()
        relative_coords[:, :, 0] += window_size - 1
        relative_coords[:, :, 1] += window_size - 1
        relative_coords[:, :, 0] *= 2 * window_size - 1
        relative_position_index = relative_coords.sum(-1)
        self.register_buffer("relative_position_index", relative_position_index)

        nn.init.trunc_normal_(self.relative_position_bias_table, std=.02)

        # For attention masking in shifted windows
        self.attn_mask = None

    def _create_attention_mask(self, H, W):
        """Create attention mask for shifted window attention"""
        if self.shift_size == 0:
            return None

        img_mask = torch.zeros((1, H, W, 1))
        h_slices = (slice(0, -self.window_size),
                    slice(-self.window_size, -self.shift_size),
                    slice(-self.shift_size, None))
        w_slices = (slice(0, -self.window_size),
                    slice(-self.window_size, -self.shift_size),
                    slice(-self.shift_size, None))
        cnt = 0
        for h in h_slices:
            for w in w_slices:
                img_mask[:, h, w, :] = cnt
                cnt += 1

        # Partition mask into windows
        mask_windows = self.window_partition(img_mask.permute(0, 3, 1, 2), self.window_size)
        mask_windows = mask_windows.view(-1, self.window_size * self.window_size)
        attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
        attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(
            attn_mask == 0, float(0.0)
        )

        return attn_mask

    def _create_boundary_bias(self, boundary_windows):
        """
        Create attention bias that discourages attention across character boundaries
        boundary_windows: (num_windows, window_size^2, 1)
        """
        # Pairwise boundary strength: high values when crossing boundaries
        boundary_diff = torch.abs(
            boundary_windows.unsqueeze(2) - boundary_windows.unsqueeze(1)
        )  # (num_windows, window_size^2, window_size^2, 1)

        # Strong penalty when crossing high boundaries
        boundary_penalty = -10.0 * boundary_diff.squeeze(-1).unsqueeze(1)  # Add head dimension

        return boundary_penalty

    def forward(self, x):
        B, C, H, W = x.shape
        r = x
        # B(x) = sigmoid(logits); during training the logits are supervised by
        # the edge map of the intact image (auxiliary loss, not an input)
        self.boundary_logits = self.boundary_predictor(x)  # B x 1 x H x W
        boundary_map = torch.sigmoid(self.boundary_logits)

        # Pad if needed
        pad_l = pad_t = 0
        pad_r = (self.window_size - W % self.window_size) % self.window_size
        pad_b = (self.window_size - H % self.window_size) % self.window_size
        x = F.pad(x, (pad_l, pad_r, pad_t, pad_b))
        boundary_map = F.pad(boundary_map, (pad_l, pad_r, pad_t, pad_b))
        _, _, Hp, Wp = x.shape

        # Create or update attention mask for shifted windows (cached per padded size)
        if self.shift_size > 0 and (self.attn_mask is None or getattr(self, '_mask_size', None) != (Hp, Wp)
                                    or self.attn_mask.device != x.device):
            self.attn_mask = self._create_attention_mask(Hp, Wp).to(x.device)
            self._mask_size = (Hp, Wp)

        # Cyclic shift (if needed)
        if self.shift_size > 0:
            shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(2, 3))
            shifted_boundary = torch.roll(boundary_map, shifts=(-self.shift_size, -self.shift_size), dims=(2, 3))
        else:
            shifted_x = x
            shifted_boundary = boundary_map

        # Partition into windows
        x_windows = self.window_partition(shifted_x, self.window_size)
        x_windows = x_windows.view(-1, self.window_size * self.window_size, C)

        # Partition the boundary map, shifted as the features
        boundary_windows = self.window_partition(shifted_boundary, self.window_size)
        boundary_windows = boundary_windows.view(-1, self.window_size * self.window_size, 1)

        # Create boundary-aware attention bias
        boundary_attn_bias = self._create_boundary_bias(boundary_windows)

        # QKV projection
        qkv = self.qkv(x_windows).reshape(
            -1, self.window_size * self.window_size, 3, self.num_heads, self.head_dim
        )
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        # Attention
        q = q * self.scale
        attn = q @ k.transpose(-2, -1)

        # Add relative position bias
        relative_position_bias = self.relative_position_bias_table[
            self.relative_position_index.view(-1)
        ].view(self.window_size * self.window_size, self.window_size * self.window_size, -1)
        relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()
        attn = attn + relative_position_bias.unsqueeze(0)

        attn = attn + boundary_attn_bias * self.boundary_bias_weight

        # Apply attention mask for shifted windows
        if self.attn_mask is not None:
            nW = self.attn_mask.shape[0]
            attn = attn.view(B, nW, self.num_heads, self.window_size * self.window_size, self.window_size * self.window_size)
            attn = attn + self.attn_mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, self.window_size * self.window_size, self.window_size * self.window_size)

        # Positions added by the padding (e.g. the 8-row bottleneck map in 16 x 16
        # windows) are excluded from the attention as keys
        if pad_r > 0 or pad_b > 0:
            N = self.window_size * self.window_size
            valid = torch.zeros((1, 1, Hp, Wp), device=x.device)
            valid[:, :, :H, :W] = 1.0
            if self.shift_size > 0:
                valid = torch.roll(valid, shifts=(-self.shift_size, -self.shift_size), dims=(2, 3))
            valid = self.window_partition(valid, self.window_size).reshape(-1, N)   # (nW, N)
            key_mask = (1.0 - valid) * -100.0
            attn = attn.view(B, -1, self.num_heads, N, N) + key_mask[None, :, None, None, :]
            attn = attn.view(-1, self.num_heads, N, N)

        attn = attn.softmax(dim=-1)

        # Apply attention
        x_windows = (attn @ v).transpose(1, 2).reshape(-1, self.window_size * self.window_size, C)
        x_windows = self.proj(x_windows)

        # Merge windows
        x_windows = x_windows.view(-1, self.window_size, self.window_size, C)
        shifted_x = self.window_reverse(x_windows, self.window_size, Hp, Wp)

        # Reverse cyclic shift
        if self.shift_size > 0:
            x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(2, 3))
        else:
            x = shifted_x

        # Remove padding
        if pad_r > 0 or pad_b > 0:
            x = x[:, :, :H, :W].contiguous()

        return x + r

    def window_partition(self, x, window_size):
        """Partition into non-overlapping windows"""
        B, C, H, W = x.shape
        x = x.view(B, C, H // window_size, window_size, W // window_size, window_size)
        windows = x.permute(0, 2, 4, 3, 5, 1).contiguous().view(-1, window_size, window_size, C)
        return windows

    def window_reverse(self, windows, window_size, H, W):
        """Reverse window partition"""
        B = int(windows.shape[0] / (H * W / window_size / window_size))
        x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
        x = x.permute(0, 5, 1, 3, 2, 4).contiguous().view(B, -1, H, W)
        return x


# ============================================================================
# GGCF: Gated Global Context Fusion
# ============================================================================

class GGCF(nn.Module):
    """Gated Global Context Fusion: Uses a gating mechanism for adaptive context fusion."""
    def __init__(self, dim, hidden_dim=None, dropout=0.1):
        super().__init__()
        hidden_dim = hidden_dim or dim * 2

        self.conv1 = nn.Conv2d(dim, hidden_dim, 1)
        self.dwconv = nn.Conv2d(hidden_dim, hidden_dim, 3, padding=1, groups=hidden_dim)
        self.conv2 = nn.Conv2d(hidden_dim, dim, 1)
        self.dropout = nn.Dropout(dropout)

        # Global context extraction
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.gmp = nn.AdaptiveMaxPool2d(1)
        self.context_proj = nn.Conv2d(hidden_dim * 2, hidden_dim, 1)

        self.fusion_gate = nn.Sequential(
            nn.Conv2d(hidden_dim * 2, hidden_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        residual = x
        x = self.conv1(x)
        x_local = F.gelu(x)

        # Extract global context
        gap_feat = self.gap(x_local)
        gmp_feat = self.gmp(x_local)
        global_context = torch.cat([gap_feat, gmp_feat], dim=1)
        global_context = self.context_proj(global_context)

        broadcasted_context = global_context.expand_as(x_local)
        gate_input = torch.cat([x_local, broadcasted_context], dim=1)
        fusion_gate = self.fusion_gate(gate_input)

        x_fused = x_local * (1 - fusion_gate) + broadcasted_context * fusion_gate

        x = self.dwconv(x_fused)
        x = F.gelu(x)
        x = self.conv2(x)
        x = self.dropout(x)

        return x + residual


# ============================================================================
# AFM: Adaptive Feature Masking
# ============================================================================

class AFM(nn.Module):
    """AFM: Adaptive Feature Masking - Feature Masking in Mixer"""
    def __init__(self, dim):
        super().__init__()
        self.conv = nn.Conv2d(dim, dim, 3, padding=1)
        self.mask_gen = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(dim, dim, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        feat = self.conv(x)
        mask = self.mask_gen(x)
        return feat * mask + x


# ============================================================================
# FDS: Frequency Decomposition Stage
# ============================================================================

class FDS(nn.Module):
    """FDS: Frequency Decomposition Stage - H-L and L-H frequency processing"""
    def __init__(self, dim):
        super().__init__()
        # High to Low frequency transformation
        self.h2l = nn.Sequential(
            nn.Conv2d(dim, dim // 2, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(dim // 2, dim, 1)
        )

        # Low to High frequency transformation
        self.l2h = nn.Sequential(
            nn.Conv2d(dim, dim // 2, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(dim // 2, dim, 1)
        )

        self.gap = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        # Extract low frequency (global smooth features)
        x_low = self.gap(x)

        # Extract high frequency (detail features)
        x_high = x - F.interpolate(F.avg_pool2d(x, 3, 1, 1), size=x.shape[2:], mode='bilinear')

        # Cross-frequency enhancement
        h2l = self.h2l(x_high)
        l2h = F.interpolate(self.l2h(x_low), size=x.shape[2:], mode='bilinear')

        return x_high + l2h, x_low.expand_as(x) + h2l


# ============================================================================
# CAM: Channel Attention Module
# ============================================================================

class CAM(nn.Module):
    """CAM: Channel Attention Module for feature recalibration"""
    def __init__(self, dim, reduction=16):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(dim, dim // reduction, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(dim // reduction, dim, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.fc(self.gap(x))


# ============================================================================
# FFM: Frequency Feature Mixer
# ============================================================================

class FFM(nn.Module):
    """FFM: Frequency Feature Mixer - Feature Mixing of Mixer"""
    def __init__(self, dim):
        super().__init__()
        self.spatial_mix = nn.Conv2d(dim, dim, 3, padding=1, groups=dim)
        self.channel_mix = nn.Conv2d(dim, dim, 1)
        self.cam = CAM(dim)

    def forward(self, x_h, x_l):
        # Mix high and low frequency features
        x = self.spatial_mix(x_h + x_l)
        x = self.channel_mix(x)
        x = self.cam(x) * x
        return x + x_h + x_l


# ============================================================================
# SRB: Spatial Restoration Block
# ============================================================================

class SRB(nn.Module):
    """
    SRB: Spatial Restoration Block
    - CA-SSM: Content-Adaptive State-Space Model
    - CG-CSWA: Character-Guided Cyclic Shift Window Attention
    - GGCF: Gated Global Context Fusion
    """
    def __init__(self, dim, num_heads=8, window_size=7, use_shift=False):
        super().__init__()
        # Content-Adaptive SSM for long-range dependencies along the reading axis
        self.ca_ssm = CA_SSM(dim)

        # Character-Guided Cyclic Shift Window Attention
        shift_size = window_size // 2 if use_shift else 0
        self.cg_cswa = CG_CSWA(dim, num_heads, window_size, shift_size)

        # Global Context Fusion
        self.ggcf = GGCF(dim)

        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.norm3 = nn.LayerNorm(dim)

    def forward(self, x):
        # Long-range with CA-SSM
        x = x + self.ca_ssm(self.apply_norm(x, self.norm1))

        # Character-guided attention with CG-CSWA
        x = x + self.cg_cswa(self.apply_norm(x, self.norm2))

        # Global context fusion
        x = x + self.ggcf(self.apply_norm(x, self.norm3))

        return x

    def apply_norm(self, x, norm):
        x = rearrange(x, 'b c h w -> b h w c')
        x = norm(x)
        x = rearrange(x, 'b h w c -> b c h w')
        return x


# ============================================================================
# FRB: Frequency Restoration Block
# ============================================================================

class FRB(nn.Module):
    """FRB: Frequency Restoration Block - AFM + FDS + style FiLM + FFM"""
    def __init__(self, dim, style_dim=None):
        super().__init__()
        self.afm = AFM(dim)
        self.fds = FDS(dim)
        self.ffm = FFM(dim)
        # Style-informed modulation of the high-frequency component (bottleneck
        # and decoder blocks). Zero-initialised, i.e. identity at the start.
        self.film = None
        if style_dim is not None:
            self.film = nn.Linear(style_dim, 2 * dim)
            nn.init.zeros_(self.film.weight)
            nn.init.zeros_(self.film.bias)

    def forward(self, x, z=None):
        r = x
        # Adaptive feature masking
        x_masked = self.afm(x)

        # Frequency decomposition
        x_high, x_low = self.fds(x_masked)

        # Style-informed modulation: h_high <- (1 + gamma(z)) * h_high + beta(z)
        if self.film is not None and z is not None:
            gamma, beta = self.film(z).chunk(2, dim=-1)
            x_high = (1 + gamma[:, :, None, None]) * x_high + beta[:, :, None, None]

        # Frequency feature mixing
        x_freq = self.ffm(x_high, x_low)

        return x_freq + r


# ============================================================================
# HRB: Hierarchical Restoration Block
# ============================================================================

class HRB(nn.Module):
    """HRB: Hierarchical Restoration Block - Dual-path processing"""
    def __init__(self, dim, num_heads=8, window_size=7, use_shift=False, style_dim=None):
        super().__init__()
        # Spatial restoration path
        self.srb = SRB(dim, num_heads, window_size, use_shift)

        # Frequency restoration path (style-modulated when style_dim is given)
        self.frb = FRB(dim, style_dim)

        # Path fusion
        self.fusion = nn.Conv2d(dim * 2, dim, 1)

    def forward(self, x, z=None):
        # Spatial restoration: CA-SSM + CG-CSWA + GGCF
        x_spatial = self.srb(x)

        # Frequency restoration: AFM + FDS + (style FiLM) + FFM
        x_freq = self.frb(x, z)

        # Fuse both paths
        x_fused = self.fusion(torch.cat([x_spatial, x_freq], dim=1))

        return x_fused + x


# ============================================================================
# PRT: Patch Refinement Transformer
# ============================================================================

class PRTLayer(nn.Module):
    """PRT Layer: Single Patch Refinement Transformer Layer"""
    def __init__(self, dim, num_heads=8, mlp_ratio=4.):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, int(dim * mlp_ratio)),
            nn.GELU(),
            nn.Linear(int(dim * mlp_ratio), dim)
        )

    def forward(self, x):
        # Self-attention for patch relationships
        x_norm = self.norm1(x)
        x = x + self.attn(x_norm, x_norm, x_norm)[0]

        # MLP for patch refinement
        x = x + self.mlp(self.norm2(x))

        return x


class PRT(nn.Module):
    """PRT: Patch Refinement Transformer for fine-grained detail enhancement"""
    def __init__(self, dim, num_heads=8, mlp_ratio=4., num_layers=2):
        super().__init__()
        self.layers = nn.ModuleList([
            PRTLayer(dim, num_heads, mlp_ratio) for _ in range(num_layers)
        ])

    def forward(self, x):
        r = x
        for layer in self.layers:
            x = layer(x)
        return x + r


# ============================================================================
# DSH: Damage Segmentation Head / RWSE: Reliability-Weighted Style Encoder
# ============================================================================

class DSH(nn.Module):
    """Damage Segmentation Head: logits of the per-pixel damage probability D_hat.
    Supervised during training by the damage map D* computed from each training
    pair (corrupted, intact); it never receives a mask as input."""
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(dim, dim, 1),
            nn.GELU(),
            nn.Conv2d(dim, dim, 3, padding=1, groups=dim),
            nn.GELU(),
            nn.Conv2d(dim, 1, 1),
        )

    def forward(self, f):
        return self.net(f)


class RWSE(nn.Module):
    """Reliability-Weighted variational Style Encoder.
    Pools the mean and standard deviation of projected features over the pixels
    predicted as intact (weights rho = 1 - D_hat, gradient stopped) and maps them
    to a Gaussian style code z ~ N(mu, sigma^2). z is sampled with the
    reparameterisation trick during training and set to mu at inference."""
    def __init__(self, dim, style_dim=64, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.proj = nn.Conv2d(dim, style_dim, 1)
        self.mlp = nn.Sequential(
            nn.Linear(2 * style_dim, 2 * style_dim),
            nn.GELU(),
            nn.Linear(2 * style_dim, 2 * style_dim),
        )

    def forward(self, f, damage_logits):
        phi = self.proj(f)
        rho = 1.0 - torch.sigmoid(damage_logits.detach())       # reliability
        w = rho / (rho.sum(dim=(2, 3), keepdim=True) + self.eps)
        mean = (w * phi).sum(dim=(2, 3))
        var = (w * (phi - mean[:, :, None, None]) ** 2).sum(dim=(2, 3))
        std = torch.sqrt(var + self.eps)
        mu, logvar = self.mlp(torch.cat((mean, std), dim=1)).chunk(2, dim=1)
        logvar = logvar.clamp(-10.0, 10.0)
        if self.training:
            z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        else:
            z = mu
        return z, mu, logvar


# ============================================================================
# MFSSNet
# ============================================================================

# Switches stored in the 'model_config' of checkpoints written by the version of
# the code that included the ablation study, with their value for the full model.
FULL_MODEL_SWITCHES = dict(use_cassm=True, use_cgcswa=True, use_ggcf=True, use_prt=True,
                           boundary_guidance=True, use_dsh=True, style_mode='variational',
                           ssm_recurrence=True)


class MFSSNet(nn.Module):
    """
    MFSSNet: A MULTI-LEVEL FLEXIBLE STYLE-INFORMED SEGMENTATION-AWARE PERSPECTIVE

    Novel Contributions:
    1. CA-SSM: Content-Adaptive State-Space Model (bidirectional selective scan)
    2. CG-CSWA: Character-Guided Cyclic Shift Window Attention
    3. HRB: Hierarchical dual-path processing (spatial + frequency)
    4. DSH + RWSE: damage segmentation head and reliability-weighted variational
       style encoder, whose style code modulates the bottleneck and decoder
    5. PRT: Patch Refinement Transformer for final detail enhancement
    """
    def __init__(self, in_channels=3, out_channels=3, dim=64,
                 num_blocks=[2, 2, 2, 2],
                 num_heads=[4, 8, 16, 32],
                 window_sizes=[8, 8, 8, 8],
                 patch_size=4,
                 num_layers=4,
                 style_dim=64):
        super().__init__()

        self.patch_size = patch_size

        # SFE: Shallow Feature Extraction
        self.sfe = nn.Conv2d(in_channels, dim, 3, padding=1)

        # HFE: Hierarchical Feature Extraction (Encoder)
        self.hfe_blocks = nn.ModuleList()
        self.downsamplers = nn.ModuleList()

        current_dim = dim
        for num, heads, ws in zip(num_blocks[:-1], num_heads[:-1], window_sizes[:-1]):
            blocks = nn.ModuleList([
                HRB(current_dim, heads, ws, use_shift=(j % 2 == 1))
                for j in range(num)
            ])
            self.hfe_blocks.append(blocks)
            self.downsamplers.append(nn.Conv2d(current_dim, current_dim * 2, 3, stride=2, padding=1))
            current_dim = current_dim * 2

        # BNE: Bottleneck Enhancement (style-modulated)
        self.bne = nn.ModuleList([
            HRB(current_dim, num_heads[-1], window_sizes[-1], use_shift=(j % 2 == 1), style_dim=style_dim)
            for j in range(num_blocks[-1])
        ])

        # HFR: Hierarchical Feature Reconstruction (Decoder, style-modulated)
        self.hfr_blocks = nn.ModuleList()
        self.upsamplers = nn.ModuleList()
        self.skip_fusions = nn.ModuleList()

        for num, heads, ws in zip(reversed(num_blocks[:-1]), reversed(num_heads[:-1]), reversed(window_sizes[:-1])):
            self.upsamplers.append(nn.ConvTranspose2d(current_dim, current_dim // 2, 2, stride=2))
            current_dim = current_dim // 2
            self.skip_fusions.append(nn.Conv2d(current_dim * 2, current_dim, 1))

            blocks = nn.ModuleList([
                HRB(current_dim, heads, ws, use_shift=(j % 2 == 1), style_dim=style_dim)
                for j in range(num)
            ])
            self.hfr_blocks.append(blocks)

        # DSH and RWSE, both applied to the output of the first encoder stage
        self.dsh = DSH(dim)
        self.rwse = RWSE(dim, style_dim)

        # PRT: Patch Refinement Transformer
        patch_dim = patch_size * patch_size * dim
        self.prt = PRT(patch_dim, num_heads=8, mlp_ratio=4., num_layers=num_layers)

        # DFR: Deep Feature Reconstruction
        self.dfr = nn.Sequential(
            nn.Conv2d(dim, dim // 2, 7, padding=3),
            nn.GELU(),
            nn.Conv2d(dim // 2, out_channels, 3, padding=1),
            nn.Sigmoid()
        )

    @classmethod
    def from_model_config(cls, model_config):
        """Builds the model from the 'model_config' stored in a checkpoint. The
        switches written by the ablation version of the code are accepted when
        they describe the full model (the only one defined here)."""
        kwargs = dict(model_config)
        for key, full_value in FULL_MODEL_SWITCHES.items():
            value = kwargs.pop(key, full_value)
            if value != full_value:
                raise ValueError(f"checkpoint of an ablation variant ({key}={value!r}): "
                                 "only the full MFSSNet is defined in this code")
        return cls(**kwargs)

    def forward(self, x, return_aux=False):
        # The only input is the corrupted image: no mask is given at any time.
        # SFE: Shallow Feature Extraction
        feat = self.sfe(x)

        # HFE: Hierarchical Feature Extraction
        skips = []
        for blocks, down in zip(self.hfe_blocks, self.downsamplers):
            for block in blocks:
                feat = block(feat)
            skips.append(feat)
            feat = down(feat)

        # Damage map (logits) and style code from the first encoder stage
        damage_logits = self.dsh(skips[0])
        z, mu, logvar = self.rwse(skips[0], damage_logits)

        # BNE: Bottleneck Enhancement (style-modulated)
        for block in self.bne:
            feat = block(feat, z)

        # HFR: Hierarchical Feature Reconstruction
        for blocks, up, skip_fusion, skip in zip(
            self.hfr_blocks, self.upsamplers, self.skip_fusions, reversed(skips)
        ):
            feat = up(feat)
            feat = skip_fusion(torch.cat([feat, skip], dim=1))
            for block in blocks:
                feat = block(feat, z)

        # PRT: Patch Refinement Transformer on non-overlapping patches
        B, C, H, W = feat.shape
        patch_seq = rearrange(feat, 'b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=self.patch_size, p2=self.patch_size)
        refined_seq = self.prt(patch_seq)
        refined_spatial = rearrange(refined_seq, 'b (h w) (p1 p2 c) -> b c (h p1) (w p2)',
                                    h=H // self.patch_size, w=W // self.patch_size,
                                    p1=self.patch_size, p2=self.patch_size)

        # DFR: Deep Feature Reconstruction
        output = self.dfr(refined_spatial)

        if return_aux:
            aux = {
                'damage_logits': damage_logits,
                'mu': mu,
                'logvar': logvar,
                'boundary_logits': [m.boundary_logits for m in self.modules()
                                    if isinstance(m, CG_CSWA) and m.boundary_logits is not None],
            }
            return output, aux
        return output


# ============================================================================
# Loss.py
# ============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models


# ============================================================================
# LOSS FUNCTIONS
# ============================================================================

class PerceptualLoss(nn.Module):
    """VGG-based perceptual loss (relu3_3); also returns relu1_2, relu2_2 and
    relu3_3 features for the style loss"""
    STYLE_LAYERS = (3, 8, 15)  # relu1_2, relu2_2, relu3_3 of VGG-16

    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        self.features = vgg.features[:16].eval()
        for param in self.features.parameters():
            param.requires_grad = False
        # ImageNet normalisation expected by the pretrained VGG-16 (images in [0, 1])
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def multi_features(self, x):
        x = (x - self.mean) / self.std
        feats = []
        for i, layer in enumerate(self.features):
            x = layer(x)
            if i in self.STYLE_LAYERS:
                feats.append(x)
        return feats

    def forward(self, pred, target):
        pred_feat = self.features((pred - self.mean) / self.std)
        target_feat = self.features((target - self.mean) / self.std)
        return F.mse_loss(pred_feat, target_feat)


class StrokeConnectivityLoss(nn.Module):
    """Damage-focused topological stroke loss.

    1. Ink maps. The grey levels of the intact image are split into two classes
       by Otsu's threshold; the smaller class is the text, which gives its
       polarity p (+1 for dark text, -1 for light text). With mu0 < mu1 the
       class means, t = (mu0 + mu1) / 2 and delta = max(mu1 - mu0, min_contrast),
       V = sigmoid(sharpness * p * (t - gray) / delta) for both images: a soft
       binarization, differentiable everywhere, whose slope adapts to the
       contrast of the text (V = 0.993 and 0.007 at the two class means).
    2. Soft skeletons of the ink maps, obtained by iterated grayscale erosion and
       opening (Shit et al., clDice, CVPR 2021). Erosion and dilation with a flat
       structuring element are min- and max-pooling: piecewise linear functions,
       differentiable almost everywhere, whose gradient goes to the pixel that
       attains the minimum or maximum, exactly as in a max-pooling layer.
    3. Topology precision (1 - share of the restored skeleton that exceeds the
       intact skeleton outside the intact ink) and sensitivity (share of the
       intact skeleton covered by restored ink, credited up to the intact ink
       level), combined as 1 - their harmonic mean, and computed only in a band
       around the damaged pixels D* (dilated by `band` pixels), where strokes
       have to be reconnected; on the whole image when an image has no damaged
       pixel. For binary maps, both reduce to the measures of clDice; for soft
       maps, both equal 1 when the restored image equals the intact one, so the
       loss is exactly 0 there and never pushes strokes beyond the intact ink.
    A broken stroke lowers the sensitivity, a spurious stroke (e.g. a residual
       scribble) lowers the precision."""

    def __init__(self, iters=10, sharpness=10.0, min_contrast=0.1, band=3, smooth=1.0, levels=256):
        super().__init__()
        self.iters, self.sharpness, self.min_contrast = iters, sharpness, min_contrast
        self.band, self.smooth, self.levels = band, smooth, levels

    @staticmethod
    def erode(x):
        """Grayscale erosion (minimum) over a 3 x 3 cross"""
        p1 = -F.max_pool2d(-x, (3, 1), (1, 1), (1, 0))
        p2 = -F.max_pool2d(-x, (1, 3), (1, 1), (0, 1))
        return torch.min(p1, p2)

    @staticmethod
    def dilate(x):
        """Grayscale dilation (maximum) over a 3 x 3 square"""
        return F.max_pool2d(x, 3, 1, 1)

    def soft_skeleton(self, x):
        skel = F.relu(x - self.dilate(self.erode(x)))
        for _ in range(self.iters):
            x = self.erode(x)
            delta = F.relu(x - self.dilate(self.erode(x)))
            skel = skel + F.relu(delta - skel * delta)
        return skel

    def text_classes(self, gray):
        """Otsu split of the grey levels of each image (sort-based, deterministic):
        threshold t between the class means, polarity of the smaller class and
        contrast between the class means, each of shape (B, 1, 1, 1)."""
        x = gray.flatten(1).sort(dim=1).values
        n = x.shape[1]
        levels = torch.linspace(0, 1, self.levels, device=x.device, dtype=x.dtype).expand(x.shape[0], -1).contiguous()
        count = torch.searchsorted(x, levels, right=True)              # pixels <= level
        csum = torch.cat([torch.zeros_like(x[:, :1]), x.cumsum(dim=1)], dim=1)
        w0, m0, mt = count / n, csum.gather(1, count) / n, csum[:, -1:] / n
        between = (mt * w0 - m0) ** 2 / (w0 * (1 - w0)).clamp_min(1e-12)
        k = between.argmax(dim=1, keepdim=True)
        w, m = w0.gather(1, k), m0.gather(1, k)
        mu0, mu1 = m / w.clamp_min(1e-12), (mt - m) / (1 - w).clamp_min(1e-12)
        polarity = torch.where(w <= 0.5, torch.ones_like(w), -torch.ones_like(w))
        shape = (-1, 1, 1, 1)
        return ((mu0 + mu1) / 2).view(shape), polarity.view(shape), (mu1 - mu0).view(shape)

    def ink(self, gray, threshold, polarity, contrast):
        return torch.sigmoid(self.sharpness * polarity * (threshold - gray) / contrast.clamp_min(self.min_contrast))

    def forward(self, pred, target, damage=None):
        with torch.autocast(device_type=pred.device.type, enabled=False):
            pred, target = pred.float(), target.float()
            g_r, g_t = pred.mean(dim=1, keepdim=True), target.mean(dim=1, keepdim=True)
            t, p, c = self.text_classes(g_t.detach())
            v_r, v_t = self.ink(g_r, t, p, c), self.ink(g_t, t, p, c)
            s_r, s_t = self.soft_skeleton(v_r), self.soft_skeleton(v_t)
            if damage is None:
                w = torch.ones_like(v_r)
            else:
                w = F.max_pool2d(damage.float(), 2 * self.band + 1, 1, self.band)
                empty = w.flatten(1).sum(dim=1) == 0
                w = torch.where(empty.view(-1, 1, 1, 1), torch.ones_like(w), w)
            dims = (1, 2, 3)
            excess = (w * F.relu(s_r - s_t) * (1 - v_t)).sum(dims)
            t_prec = 1 - excess / ((w * s_r).sum(dims) + self.smooth)
            t_sens = ((w * s_t * torch.minimum(v_r, v_t)).sum(dims) + self.smooth) / ((w * s_t * v_t).sum(dims) + self.smooth)
            return (1.0 - 2.0 * t_prec * t_sens / (t_prec + t_sens)).mean()


class MFSSNetLoss(nn.Module):
    """Comprehensive loss function for MFSSNet model"""
    def __init__(self, alpha_pixel=1.0, alpha_perceptual=0.5, alpha_edge=0.2, alpha_char=0.3, alpha_stroke=0.3,
                 alpha_style=120.0, alpha_seg=0.5, alpha_bnd=0.2, beta_kl=1e-3):
        super().__init__()
        self.alpha_pixel = alpha_pixel
        self.alpha_perceptual = alpha_perceptual
        self.alpha_edge = alpha_edge
        self.alpha_char = alpha_char
        self.alpha_stroke = alpha_stroke
        self.alpha_style = alpha_style
        self.alpha_seg = alpha_seg
        self.alpha_bnd = alpha_bnd
        self.beta_kl = beta_kl

        self.vgg = PerceptualLoss()
        self.stroke_loss = StrokeConnectivityLoss()

        self.register_buffer('sobel_x', torch.tensor([[[[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]]]).float())
        self.register_buffer('sobel_y', torch.tensor([[[[-1, -2, -1], [0, 0, 0], [1, 2, 1]]]]).float())

    def edge_loss(self, pred, target):
        pred_edges_x = F.conv2d(pred.mean(dim=1, keepdim=True), self.sobel_x, padding=1)
        pred_edges_y = F.conv2d(pred.mean(dim=1, keepdim=True), self.sobel_y, padding=1)
        target_edges_x = F.conv2d(target.mean(dim=1, keepdim=True), self.sobel_x, padding=1)
        target_edges_y = F.conv2d(target.mean(dim=1, keepdim=True), self.sobel_y, padding=1)

        return F.l1_loss(pred_edges_x, target_edges_x) + F.l1_loss(pred_edges_y, target_edges_y)

    def character_consistency_loss(self, pred, target):
        patch_size = 16
        unfold = nn.Unfold(kernel_size=patch_size, stride=patch_size // 2)

        pred_patches = unfold(pred)
        target_patches = unfold(target)

        pred_patches_norm = F.normalize(pred_patches, dim=1)
        target_patches_norm = F.normalize(target_patches, dim=1)

        similarity = torch.sum(pred_patches_norm * target_patches_norm, dim=1)
        return 1.0 - similarity.mean()

    # ---- style loss (Gram matrices of VGG-16 relu1_2, relu2_2, relu3_3) ----
    @staticmethod
    def gram(f):
        b, c, h, w = f.shape
        f = f.reshape(b, c, h * w)
        return torch.bmm(f, f.transpose(1, 2)) / (c * h * w)

    def style_loss(self, pred_feats, target_feats):
        # Gram matrices are computed in float32 (sums over H*W overflow in fp16)
        with torch.autocast(device_type=pred_feats[0].device.type, enabled=False):
            return sum(F.l1_loss(self.gram(p.float()), self.gram(t.float()))
                       for p, t in zip(pred_feats, target_feats))

    # ---- auxiliary losses (training only) ----
    def edge_target(self, target):
        """E*: Sobel gradient magnitude of the intact image, normalised to [0, 1]"""
        gray = target.float().mean(dim=1, keepdim=True)
        gx = F.conv2d(gray, self.sobel_x.float(), padding=1)
        gy = F.conv2d(gray, self.sobel_y.float(), padding=1)
        mag = torch.sqrt(gx ** 2 + gy ** 2 + 1e-12)
        return mag / (mag.amax(dim=(2, 3), keepdim=True) + 1e-6)

    def boundary_loss(self, boundary_logits, target):
        e = self.edge_target(target)
        losses = [F.binary_cross_entropy_with_logits(
                      b.float(), F.adaptive_max_pool2d(e, b.shape[-2:]))
                  for b in boundary_logits]
        return torch.stack(losses).mean()

    @staticmethod
    def damage_loss(logits, target, smooth=1.0):
        logits, target = logits.float(), target.float()
        bce = F.binary_cross_entropy_with_logits(logits, target)
        p = torch.sigmoid(logits)
        inter = (p * target).sum(dim=(1, 2, 3))
        dice = 1 - (2 * inter + smooth) / (p.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) + smooth)
        return bce + dice.mean()

    @staticmethod
    def kl_loss(mu, logvar):
        mu, logvar = mu.float(), logvar.float()
        return 0.5 * (mu ** 2 + logvar.exp() - logvar - 1).sum(dim=1).mean()

    def forward(self, pred, target, aux=None, damage_target=None):
        # A term whose weight is 0 is not computed.
        pixel_loss = F.l1_loss(pred, target)
        total_loss = self.alpha_pixel * pixel_loss
        logs = {'pixel': pixel_loss.item()}
        pred_feats = self.vgg.multi_features(pred)
        target_feats = self.vgg.multi_features(target)
        terms = {
            'perceptual': (self.alpha_perceptual, lambda: F.mse_loss(pred_feats[-1], target_feats[-1])),  # relu3_3
            'edge': (self.alpha_edge, lambda: self.edge_loss(pred, target)),
            'character': (self.alpha_char, lambda: self.character_consistency_loss(pred, target)),
            'stroke': (self.alpha_stroke, lambda: self.stroke_loss(pred, target, damage_target)),
            'style': (self.alpha_style, lambda: self.style_loss(pred_feats, target_feats)),
        }
        # Auxiliary predictions (training only; not used as inputs)
        if aux is not None:
            if aux.get('boundary_logits'):
                terms['boundary'] = (self.alpha_bnd, lambda: self.boundary_loss(aux['boundary_logits'], target))
            if aux.get('logvar') is not None:
                terms['kl'] = (self.beta_kl, lambda: self.kl_loss(aux['mu'], aux['logvar']))
            if damage_target is not None and aux.get('damage_logits') is not None:
                terms['damage'] = (self.alpha_seg, lambda: self.damage_loss(aux['damage_logits'], damage_target))
        for name, (weight, term) in terms.items():
            if weight > 0:
                value = term()
                total_loss = total_loss + weight * value
                logs[name] = value.item()

        return total_loss, logs


# ============================================================================
# Dataset.py
# ============================================================================

import hashlib
import logging
import random
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset, Subset
from torchvision import transforms

logger = logging.getLogger(__name__)


# ============================================================================
# DATASET
# ============================================================================

class BaseTextDataset(Dataset):
    """Pairs (corrupted, intact) of the TII-ST / TII-HT benchmarks.
    No mask is read: the network never receives one, and the damage target
    D* is computed from the pair itself."""
    def __init__(self, img_size=(64, 256), augment=False, config=None):
        self.img_size = img_size
        self.augment = augment
        self.config = config
        self.image_pairs = []
        # Threshold of the damage target D* (see __getitem__)
        self.damage_tau = getattr(config, 'damage_tau', 0.1)
        # Gaussian-noise augmentation of the training input. 0 = benchmark
        # protocol (same training data as the compared methods); > 0 only for
        # the model used in the separate robustness study.
        self.noise_prob = getattr(config, 'noise_prob', 0.0)
        # Noise added to TEST inputs, for the separate robustness study only.
        # 0 = benchmark protocol (test images used as distributed).
        self.test_noise_sigma = getattr(config, 'test_noise_sigma', 0.0)

        base_transforms = [transforms.Resize(img_size), transforms.ToTensor()] if img_size else [transforms.ToTensor()]
        self.transform = transforms.Compose(base_transforms)

    def apply_corruption(self, image):
        """Gaussian noise, sigma ~ U[0.05, 0.15] (training augmentation)"""
        noise = torch.randn_like(image) * random.uniform(0.05, 0.15)
        return torch.clamp(image + noise, 0, 1)

    def __len__(self):
        return len(self.image_pairs)

    def __getitem__(self, idx):
        pair = self.image_pairs[idx]
        corrupted = self.transform(Image.open(pair['corrupted']).convert('RGB'))
        clean = self.transform(Image.open(pair['clean']).convert('RGB'))

        # Damage target D* (training-only supervision of the damage head):
        # pixels where the corrupted input differs from the intact image,
        # computed before any noise; no annotation needed.
        damage = ((corrupted - clean).abs().amax(dim=0, keepdim=True) > self.damage_tau).float()

        if self.augment and self.noise_prob > 0 and random.random() < self.noise_prob:
            corrupted = self.apply_corruption(corrupted)
        elif not self.augment and self.test_noise_sigma > 0:
            # Deterministic per image, so that every method sees the same noisy input
            g = torch.Generator().manual_seed(idx)
            corrupted = torch.clamp(corrupted + self.test_noise_sigma * torch.randn(corrupted.shape, generator=g), 0, 1)

        return {
            'corrupted': torch.clamp(corrupted, 0, 1),
            'clean': torch.clamp(clean, 0, 1),
            'damage': damage,
            'image_id': pair['id'],
            'name': pair['name'],
            'source': pair['source'],
            'dataset_type': pair['type']
        }


def _as_list(dirs):
    if dirs is None:
        return []
    return [dirs] if isinstance(dirs, (str, Path)) else list(dirs)


class MergedTextDataset(BaseTextDataset):
    """TII-ST (scene text: synthetic + real) and/or TII-HT (handwritten) pairs.
    `synthetic_dir` and `handwritten_dir` accept one folder or a list of folders,
    e.g. every source of the official TII-ST training set."""

    def __init__(self, synthetic_dir=None, handwritten_dir=None, split='train',
                 augment=False, config=None, img_size=None):
        super().__init__(img_size, augment, config)
        self.counts = {}
        for d in _as_list(synthetic_dir):
            self._load_synthetic(Path(d), split)
        for d in _as_list(handwritten_dir):
            self._load_handwritten(Path(d))

    def _add(self, img_path, clean_path, root, kind):
        self.image_pairs.append({
            'id': f"{kind}_{root.name}_{img_path.stem}",
            'name': img_path.name,
            'source': root.name,
            'corrupted': str(img_path),
            'clean': str(clean_path),
            'type': kind
        })
        self.counts[str(root)] = self.counts.get(str(root), 0) + 1

    def _load_synthetic(self, root_dir, split):
        if split == 'train':
            incomplete_dir, complete_dir = root_dir / 'train_incomplete', root_dir / 'train_f'
        else:
            incomplete_dir, complete_dir = root_dir / 't_incomplete', root_dir / 't_f'
        if not incomplete_dir.exists() and (root_dir / 'Corrupted_Image').exists():
            # the synthetic part of TII-ST (syth80k) is released with the TII-HT layout
            incomplete_dir, complete_dir = root_dir / 'Corrupted_Image', root_dir / 'Intact_Image'
        if not incomplete_dir.exists():
            logger.warning(f"Scene-text directory not found: {incomplete_dir}")
            return
        missing = 0
        for img_path in sorted(incomplete_dir.glob('*.png')):
            complete_path = complete_dir / img_path.name
            if complete_path.exists():
                self._add(img_path, complete_path, root_dir, 'scene')
            else:
                missing += 1
        if missing:
            logger.warning(f"{missing} corrupted images without intact counterpart in {root_dir}")

    def _load_handwritten(self, root_dir):
        corrupted_dir, intact_dir = root_dir / 'Corrupted_Image', root_dir / 'Intact_Image'
        if not corrupted_dir.exists():
            logger.warning(f"Handwritten directory not found: {corrupted_dir}")
            return
        missing = 0
        for img_path in sorted(corrupted_dir.glob('*.png')):
            intact_path = intact_dir / img_path.name
            if intact_path.exists():
                self._add(img_path, intact_path, root_dir, 'handwritten')
            else:
                missing += 1
        if missing:
            logger.warning(f"{missing} corrupted images without intact counterpart in {root_dir}")


# ============================================================================
# BENCHMARK PROTOCOL (Zhu et al., AAAI 2024): splits, seeds, sanity checks
# ============================================================================

# Official split sizes of the TII benchmarks
OFFICIAL_COUNTS = {
    'st_train': 80000 + 4877,   # synthetic + real (ICDAR 2013/2015/2017)
    'st_test': 1599,            # real
    'ht_train': 38578,          # IAM
    'ht_test': 1600,
}


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def check_counts(dataset, key, strict=True):
    """Compare the number of loaded pairs with the official split size."""
    n, expected = len(dataset), OFFICIAL_COUNTS[key]
    msg = f"{key}: {n} pairs loaded (official: {expected}); per folder: {dataset.counts}"
    if n != expected:
        if strict:
            raise ValueError(msg + " -> the split differs from the benchmark; "
                                   "add the missing source folders or set config.strict_counts = False")
        logger.warning(msg)
    else:
        logger.info(msg)


def split_indices(n, val_fraction=0.02, seed=42):
    """Deterministic train/validation split of the TRAINING set (the official
    test set is never used for training or model selection)."""
    idx = list(range(n))
    random.Random(seed).shuffle(idx)
    n_val = max(1, int(round(val_fraction * n)))
    return sorted(idx[n_val:]), sorted(idx[:n_val])


def split_manifest(train_set, val_set, test_set):
    """Images of each split (source/name) with a SHA-256 fingerprint of each
    list, saved with every run so that the exact splits can be published."""
    def ids(ds):
        base, idx = (ds.dataset, ds.indices) if isinstance(ds, Subset) else (ds, range(len(ds)))
        return [f"{base.image_pairs[i]['source']}/{base.image_pairs[i]['name']}" for i in idx]
    out = {}
    for name, ds in (('train', train_set), ('validation', val_set), ('test', test_set)):
        lst = ids(ds)
        out[name] = {'n_images': len(lst), 'sha256': hashlib.sha256('\n'.join(lst).encode()).hexdigest(),
                     'images': lst}
    return out


def _md5(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()


def build_benchmark_datasets(config):
    """Training / validation / test sets of ONE benchmark (config.train_on =
    'st' or 'ht'), as in the protocol of the compared methods."""
    kind = config.train_on
    dirs = {'st': (config.st_train_dirs, config.st_test_dirs),
            'ht': (config.ht_train_dirs, config.ht_test_dirs)}[kind]
    kw = dict(img_size=config.img_size, config=config)
    as_arg = (lambda d: dict(synthetic_dir=d)) if kind == 'st' else (lambda d: dict(handwritten_dir=d))

    train_aug = MergedTextDataset(split='train', augment=True, **as_arg(dirs[0]), **kw)
    train_plain = MergedTextDataset(split='train', augment=False, **as_arg(dirs[0]), **kw)
    test_set = MergedTextDataset(split='test', augment=False, **as_arg(dirs[1]), **kw)
    strict = getattr(config, 'strict_counts', True)
    check_counts(train_aug, f'{kind}_train', strict)
    check_counts(test_set, f'{kind}_test', strict)

    tr_idx, va_idx = split_indices(len(train_aug), config.val_fraction, config.seed)
    train_set = Subset(train_aug, tr_idx)
    val_set = Subset(train_plain, va_idx)

    # Guard against leakage: no test image may also be a training image
    if getattr(config, 'check_leakage', True):
        train_hashes = {_md5(p['corrupted']) for p in train_aug.image_pairs}
        overlap = sum(_md5(p['corrupted']) in train_hashes for p in test_set.image_pairs)
        if overlap:
            logger.warning(f"{overlap} test images are byte-identical to training images")
        else:
            logger.info("No test image found in the training set")
    return train_set, val_set, test_set


# ============================================================================
# JOINT TII-ST + TII-HT TRAINING SET (real pairs only)
# ============================================================================
# Only the real, distributed pairs of the two benchmarks are used: nothing is
# generated, simulated or randomly altered (no noise augmentation, no mask
# synthesis; the damage target D* is computed from the pair as in
# BaseTextDataset). Each item has exactly the keys and values of an item of
# MergedTextDataset: corrupted, clean, damage, image_id, name, source,
# dataset_type ('scene' for TII-ST, 'handwritten' for TII-HT).
# The official test sets stay SEPARATE (different protocols): a joint model is
# evaluated on TII-ST test and on TII-HT test, one at a time.

JOINT_DOMAIN = {'scene': 'st', 'handwritten': 'ht'}


class JointTextDataset(MergedTextDataset):
    """TII-ST (synthetic + real folders) and TII-HT (IAM) pairs in one dataset."""

    def __init__(self, st_dirs=None, ht_dirs=None, split='train', config=None, img_size=None, verify=True):
        if config is not None and (getattr(config, 'noise_prob', 0.0) > 0 or getattr(config, 'test_noise_sigma', 0.0) > 0):
            raise ValueError("joint training uses the real pairs as distributed: set config.noise_prob = 0 and "
                             "config.test_noise_sigma = 0")
        super().__init__(synthetic_dir=st_dirs, handwritten_dir=ht_dirs, split=split, augment=False,
                         config=config, img_size=img_size)
        ids = [p['id'] for p in self.image_pairs]
        if len(set(ids)) != len(ids):                       # two folders with the same name: make the ids unique
            seen, dup = {}, 0
            for p in self.image_pairs:
                k = p['id']
                seen[k] = seen.get(k, 0) + 1
                if seen[k] > 1:
                    p['id'] = f"{k}#{hashlib.md5(p['corrupted'].encode()).hexdigest()[:8]}"
                    dup += 1
            logger.warning(f"{dup} image ids were not unique across folders and got a path hash")
        if verify:
            self.verify_pairs()

    def verify_pairs(self):
        """Reads the header of every file (no pixel decoding): both images of a pair must open, be RGB-convertible
        and have the same size. A pair that fails is a data error: the dataset stops instead of dropping it."""
        bad = []
        for p in self.image_pairs:
            try:
                with Image.open(p['corrupted']) as a, Image.open(p['clean']) as b:
                    if a.size != b.size:
                        bad.append((p['id'], f"size {a.size} vs {b.size}"))
            except Exception as e:                              # noqa: BLE001
                bad.append((p['id'], f"{type(e).__name__}: {e}"))
        if bad:
            raise ValueError(f"{len(bad)} corrupted/intact pairs are inconsistent, e.g. {bad[:3]}")

    def domain(self, i):
        """'st' or 'ht' of pair i."""
        return JOINT_DOMAIN[self.image_pairs[i]['type']]

    def domain_indices(self):
        out = {'st': [], 'ht': []}
        for i, p in enumerate(self.image_pairs):
            out[JOINT_DOMAIN[p['type']]].append(i)
        return out

    def summary(self):
        di = self.domain_indices()
        return {'st': len(di['st']), 'ht': len(di['ht']), 'total': len(self), 'per_folder': dict(self.counts)}


def joint_split_indices(dataset, val_fraction=0.02, seed=42):
    """Deterministic validation split drawn INSIDE each benchmark (so both are represented in the validation set).
    Only the training pairs are split: the official test sets are never used for training or model selection."""
    tr, va = [], []
    for dom, idx in dataset.domain_indices().items():
        t_, v_ = split_indices(len(idx), val_fraction, seed)
        tr += [idx[i] for i in t_]
        va += [idx[i] for i in v_]
    return sorted(tr), sorted(va)


def joint_sampler(train_set, mode='none', seed=42):
    """None (default): every real pair is seen exactly once per epoch.
    'equal': a seeded WeightedRandomSampler that gives each benchmark the same share of every epoch by re-drawing
    REAL pairs (the smaller benchmark is seen more than once per epoch; no image is created)."""
    if mode in (None, 'none'):
        return None
    if mode != 'equal':
        raise ValueError("mode must be 'none' or 'equal'")
    base, idx = (train_set.dataset, train_set.indices) if isinstance(train_set, Subset) else (train_set, range(len(train_set)))
    doms = [JOINT_DOMAIN[base.image_pairs[i]['type']] for i in idx]
    n = {d: doms.count(d) for d in ('st', 'ht')}
    weights = torch.tensor([1.0 / n[d] for d in doms], dtype=torch.double)
    g = torch.Generator().manual_seed(seed)
    return torch.utils.data.WeightedRandomSampler(weights, num_samples=len(doms), replacement=True, generator=g)


def build_joint_datasets(config):
    """(train_set, val_set, test_sets) of the joint ST + HT training.
    train_set / val_set: Subsets of one JointTextDataset (both benchmarks); test_sets: {'st': ..., 'ht': ...}, the two
    official test sets, each evaluated with its own protocol."""
    kw = dict(config=config, img_size=config.img_size)
    full = JointTextDataset(config.st_train_dirs, config.ht_train_dirs, split='train', **kw)
    di = full.domain_indices()
    strict = getattr(config, 'strict_counts', True)
    for dom in ('st', 'ht'):
        n, expected = len(di[dom]), OFFICIAL_COUNTS[f'{dom}_train']
        msg = f"{dom}_train: {n} pairs loaded (official: {expected})"
        if n != expected:
            if strict:
                raise ValueError(msg + " -> the split differs from the benchmark; add the missing source folders "
                                       "or set config.strict_counts = False")
            logger.warning(msg)
        else:
            logger.info(msg)
    tests = {'st': MergedTextDataset(synthetic_dir=config.st_test_dirs, split='test', augment=False, **kw),
             'ht': MergedTextDataset(handwritten_dir=config.ht_test_dirs, split='test', augment=False, **kw)}
    for dom, ds in tests.items():
        check_counts(ds, f'{dom}_test', strict)
    tr_idx, va_idx = joint_split_indices(full, config.val_fraction, config.seed)
    train_set, val_set = Subset(full, tr_idx), Subset(full, va_idx)
    if getattr(config, 'check_leakage', True):
        train_hashes = {_md5(p['corrupted']) for p in full.image_pairs}
        for dom, ds in tests.items():
            overlap = sum(_md5(p['corrupted']) in train_hashes for p in ds.image_pairs)
            if overlap:
                logger.warning(f"{dom}_test: {overlap} test images are byte-identical to joint training images")
            else:
                logger.info(f"{dom}_test: no test image found in the joint training set")
    logger.info(f"joint set: {full.summary()}; train {len(train_set)}, validation {len(val_set)}")
    return train_set, val_set, tests


# ============================================================================
# Train.py
# ============================================================================

import csv
import json
import logging
import math
import os
import time
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import torch.optim as optim
from PIL import Image
from torch.amp import autocast, GradScaler
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm


# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    """Configuration for MFSSNet training and evaluation"""
    def __init__(self):
        # Model architecture
        self.in_channels = 3
        self.out_channels = 3
        self.dim = 32
        self.num_blocks = [2, 2, 2, 2]
        self.num_heads = [2, 4, 8, 16]
        self.window_sizes = [8, 8, 16, 16]
        self.patch_size = 4
        self.num_prt_layers = 4
        self.style_dim = 64        # dimension of the style code z

        # Training (these values are the ones used; they are reported in Section 4.1)
        self.batch_size = 52
        self.epochs = 30
        self.lr = 1e-4
        self.weight_decay = 1e-4
        self.img_size = (64, 256)

        # Mixed Precision & Gradient Clipping
        self.use_amp = True  # Automatic Mixed Precision
        self.grad_clip_norm = 0.5  # Gradient clipping max norm

        # Advanced Scheduler Configuration
        self.warmup_epochs = 5  # Linear warmup epochs
        self.scheduler_t0 = 50  # CosineAnnealing: epochs until first restart
        self.scheduler_tmult = 2  # CosineAnnealing: period multiplier after restart

        # Loss weights
        self.alpha_pixel = 1.0
        self.alpha_perceptual = 0.5
        self.alpha_edge = 0.3
        self.alpha_char = 0.3
        self.alpha_stroke = 0.3
        self.alpha_style = 120.0   # Gram-matrix style loss
        self.alpha_seg = 0.5       # damage segmentation (BCE + Dice)
        self.alpha_bnd = 0.2       # boundary map supervision
        self.beta_kl = 1e-3        # KL of the variational style code
        self.damage_tau = 0.1      # threshold of the damage target D*

        # Benchmark protocol (Zhu et al., AAAI 2024)
        self.seed = 42
        self.train_on = 'ht'          # 'st' or 'ht': one model per benchmark
        self.val_fraction = 0.02      # validation split held out from the TRAINING set
        self.noise_prob = 0.0         # 0 = no augmentation (same training data as the baselines)
        self.test_noise_sigma = 0.0   # 0 = test images used as distributed
        self.strict_counts = True     # stop if a split differs from the official one
        self.check_leakage = True     # check that no test image is a training image
        # Model selection: highest validation PSNR, a criterion that does not
        # depend on the loss weights
        self.selection_metric = 'psnr'

        # Every source folder of the official splits must be listed
        # (TII-ST: synthetic + IC13 + IC15 + IC17; TII-HT: IAM)
        self.st_train_dirs = ['/kaggle/working/stt_train']
        self.st_test_dirs = ['/kaggle/working/stt_test']
        self.ht_train_dirs = ['/kaggle/working/htt_train']
        self.ht_test_dirs = ['/kaggle/working/htt_test']

        # Evaluation
        self.metrics = ['psnr', 'ssim']   # validation metrics (selection on 'psnr')

        # Paths
        self.checkpoint_dir = '/content'
        self.results_dir = './results/MFSSNet'
        self.analysis_dir = './results/analysis'
        self.manipulated_dir = './results/manipulated'

        # Resume training
        self.resume = True

    def create_directories(self):
        """Create necessary directories"""
        dirs = [
            self.checkpoint_dir,
            self.results_dir,
            self.analysis_dir,
            f"{self.manipulated_dir}/corrupted",
            f"{self.manipulated_dir}/restored",
            f"{self.manipulated_dir}/clean",
            f"{self.analysis_dir}/comprehensive",
            f"{self.analysis_dir}/lam_maps"
        ]
        for d in dirs:
            Path(d).mkdir(parents=True, exist_ok=True)


def model_kwargs(config):
    """Arguments of MFSSNet for a config."""
    return dict(in_channels=config.in_channels, out_channels=config.out_channels, dim=config.dim,
                num_blocks=config.num_blocks, num_heads=config.num_heads,
                window_sizes=config.window_sizes, patch_size=config.patch_size,
                num_layers=config.num_prt_layers, style_dim=config.style_dim)


# ============================================================================
# METRICS
# ============================================================================

class MetricsEvaluator:
    """Centralized metrics evaluation"""
    def __init__(self, metrics=None, device='cuda'):
        self.device = device
        self.metrics_names = metrics if metrics else ['psnr', 'ssim']

        # Try to import pyiqa if available
        try:
            import pyiqa
            self.metrics = {}
            for metric_name in self.metrics_names:
                try:
                    self.metrics[metric_name] = pyiqa.create_metric(metric_name, device=device)
                except Exception as e:
                    logger.warning(f"Failed to initialize {metric_name}: {e}")
        except ImportError:
            logger.warning("pyiqa not available, using basic metrics")
            self.metrics = {}

        if not self.metrics:
            logger.warning("Using fallback PSNR/SSIM computation")

    def compute_psnr(self, pred, target):
        """Compute PSNR"""
        mse = F.mse_loss(pred, target)
        if mse == 0:
            return torch.tensor(100.0)
        return 20 * torch.log10(1.0 / torch.sqrt(mse))

    def compute_ssim(self, pred, target):
        """Simplified SSIM computation"""
        C1 = 0.01 ** 2
        C2 = 0.03 ** 2

        mu_pred = F.avg_pool2d(pred, 3, 1, 1)
        mu_target = F.avg_pool2d(target, 3, 1, 1)

        sigma_pred = F.avg_pool2d(pred * pred, 3, 1, 1) - mu_pred * mu_pred
        sigma_target = F.avg_pool2d(target * target, 3, 1, 1) - mu_target * mu_target
        sigma_pred_target = F.avg_pool2d(pred * target, 3, 1, 1) - mu_pred * mu_target

        ssim_map = ((2 * mu_pred * mu_target + C1) * (2 * sigma_pred_target + C2)) / \
                   ((mu_pred ** 2 + mu_target ** 2 + C1) * (sigma_pred + sigma_target + C2))

        return ssim_map.mean()

    def compute(self, pred, target):
        # Metrics in float32: under autocast pred is float16, and pyiqa's range check
        # (x.max() < 1 + 1e-6) rounds 1 + 1e-6 to 1.0 in float16, so a pixel equal to
        # 1.0 was rejected and no PSNR/SSIM was computed.
        pred = torch.clamp(pred.float(), 0, 1)
        target = torch.clamp(target.float(), 0, 1)

        results = {}

        if self.metrics:
            for name, metric_fn in self.metrics.items():
                try:
                    if name in ['niqe', 'brisque', 'musiq']:
                        score = metric_fn(pred)
                    else:
                        score = metric_fn(pred, target)
                    results[name] = score.mean().item() if torch.is_tensor(score) else score
                except Exception as e:
                    logger.warning(f"Error computing {name}: {e}")
        else:
            # Fallback to basic metrics
            results['psnr'] = self.compute_psnr(pred, target).item()
            results['ssim'] = self.compute_ssim(pred, target).item()

        return results


# ============================================================================
# BENCHMARK EVALUATION (protocol and metric code of Zhu et al., AAAI 2024)
# ============================================================================
# - official test sets used as distributed (64x256), no added noise
# - restored images saved as 8-bit PNG under their original file names, so that
#   the official recognizers (CRNN, ASTER, MORAN / DAN, TrOCR-B, TrOCR-L) can be
#   run on them exactly as on the outputs of the compared methods
# - PSNR on RGB in [0, 255] and SSIM on RGB (11x11 Gaussian window, sigma 1.5),
#   computed from the saved PNG files and averaged over images, as in the
#   metric code released with GSDM (metrics/psnr.py, metrics/ssim.py);
#   files are paired by name.

def _gsdm_gaussian_window(window_size=11, sigma=1.5, channel=3):
    g = torch.tensor([math.exp(-(x - window_size // 2) ** 2 / float(2 * sigma ** 2))
                      for x in range(window_size)])
    g = (g / g.sum()).unsqueeze(1)
    w2d = g.mm(g.t()).float().unsqueeze(0).unsqueeze(0)
    return w2d.expand(channel, 1, window_size, window_size).contiguous()


def gsdm_psnr(a, b):
    """PSNR of GSDM's metrics/psnr.py (inputs in [0, 255]); returns 0 when the
    images are identical, as the reference code does."""
    mse = torch.mean((a.float() - b.float()) ** 2)
    if mse == 0:
        return 0.0
    return (20 * math.log10(255.0) - 10 * torch.log10(mse)).item()


def gsdm_ssim(img1, img2, window_size=11):
    """SSIM of GSDM's metrics/ssim.py (inputs in [0, 1], RGB, mean of the map)."""
    channel = img1.shape[1]
    w = _gsdm_gaussian_window(window_size, 1.5, channel).to(img1.device, img1.dtype)
    p = window_size // 2
    mu1 = F.conv2d(img1, w, padding=p, groups=channel)
    mu2 = F.conv2d(img2, w, padding=p, groups=channel)
    mu1_sq, mu2_sq, mu1_mu2 = mu1.pow(2), mu2.pow(2), mu1 * mu2
    s1 = F.conv2d(img1 * img1, w, padding=p, groups=channel) - mu1_sq
    s2 = F.conv2d(img2 * img2, w, padding=p, groups=channel) - mu2_sq
    s12 = F.conv2d(img1 * img2, w, padding=p, groups=channel) - mu1_mu2
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    ssim_map = ((2 * mu1_mu2 + C1) * (2 * s12 + C2)) / ((mu1_sq + mu2_sq + C1) * (s1 + s2 + C2))
    return ssim_map.mean().item()


def _to_png(t):
    """[0,1] tensor (3,H,W) -> 8-bit PIL image"""
    arr = (t.clamp(0, 1).mul(255).round().byte().permute(1, 2, 0).cpu().numpy())
    return Image.fromarray(arr)


@torch.no_grad()
def evaluate_benchmark(model, test_set, out_dir, device, batch_size=16, groups_csv=None):
    """Restores every image of an official test set, saves the outputs and
    computes PSNR / SSIM with the metric code of the benchmark.
    groups_csv (optional): CSV with columns name,group (e.g. corrosion pattern
    or ratio range) to report the metrics per group."""
    out_dir = Path(out_dir)
    model.eval()
    loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=2)
    for batch in loader:
        restored = torch.clamp(model(batch['corrupted'].to(device)), 0.0, 1.0).cpu()
        for i in range(restored.shape[0]):
            d = out_dir / batch['source'][i]
            d.mkdir(parents=True, exist_ok=True)
            _to_png(restored[i]).save(d / batch['name'][i])

    to_t = transforms.ToTensor()
    rows = []
    for pair in test_set.image_pairs:
        gt = Image.open(pair['clean']).convert('RGB')
        out = Image.open(out_dir / pair['source'] / pair['name']).convert('RGB')
        if out.size != gt.size:
            out = out.resize(gt.size, Image.BICUBIC)
        g, o = to_t(gt).unsqueeze(0), to_t(out).unsqueeze(0)
        rows.append({'source': pair['source'], 'name': pair['name'],
                     'psnr': gsdm_psnr(g * 255., o * 255.), 'ssim': gsdm_ssim(g, o)})

    with open(out_dir / 'per_image_metrics.csv', 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['source', 'name', 'psnr', 'ssim'])
        w.writeheader()
        w.writerows(rows)

    summary = {'n_images': len(rows),
               'psnr': float(np.mean([r['psnr'] for r in rows])),
               'ssim': float(np.mean([r['ssim'] for r in rows]))}
    if groups_csv:
        with open(groups_csv) as f:
            group_of = {r['name']: r['group'] for r in csv.DictReader(f)}
        by_group = defaultdict(list)
        for r in rows:
            if r['name'] in group_of:
                by_group[group_of[r['name']]].append(r)
        summary['groups'] = {k: {'n_images': len(v),
                                 'psnr': float(np.mean([r['psnr'] for r in v])),
                                 'ssim': float(np.mean([r['ssim'] for r in v]))}
                             for k, v in sorted(by_group.items())}
    with open(out_dir / 'summary.json', 'w') as f:
        json.dump(summary, f, indent=2)
    return summary


@torch.no_grad()
def time_inference(model, test_set, device, n_images=200, warmup=20):
    """Inference time per image (batch size 1), after warm-up; to be run for all
    timed methods on the same GPU."""
    model.eval()
    times = []
    n = min(len(test_set), n_images + warmup)
    for k in range(n):
        x = test_set[k]['corrupted'].unsqueeze(0).to(device)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        model(x)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        if k >= warmup:
            times.append(time.perf_counter() - t0)
    gpu = torch.cuda.get_device_name(0) if device.type == 'cuda' else 'cpu'
    return {'device': gpu, 'n_images': len(times),
            'mean_s': float(np.mean(times)), 'std_s': float(np.std(times))}


def mask_iou(pred, target):
    """IoU of two boolean masks (per image); None when both are empty."""
    union = (pred | target).sum()
    if union == 0:
        return None
    return float((pred & target).sum() / union)


@torch.no_grad()
def evaluate_damage_head(model, test_set, device, batch_size=16, out_csv=None):
    """Coverage of the damage and mask IoU of the damage head.
    Target: damage map D* of each (corrupted, intact) test pair (Eq. (19), tau = 0.1).
    Prediction: DSH probability > 0.5, from the corrupted image alone.
    Both are averaged over images; images whose prediction and target are both
    empty are left out of the IoU mean and counted separately. out_csv (optional):
    per-image values (source, name, coverage, iou)."""
    model.eval()
    loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=2)
    cov, ious, n_empty, rows = [], [], 0, []
    for batch in loader:
        _, aux = model(batch['corrupted'].to(device), return_aux=True)
        pred = (aux['damage_logits'] > 0).cpu()
        target = batch['damage'] > 0.5
        for i in range(pred.shape[0]):
            cov.append(float(target[i].float().mean()))
            v = mask_iou(pred[i], target[i])
            if v is None:
                n_empty += 1
            else:
                ious.append(v)
            rows.append({'source': batch['source'][i], 'name': batch['name'][i],
                         'coverage': cov[-1], 'iou': '' if v is None else v})
    if out_csv is not None:
        with open(out_csv, 'w', newline='') as f:
            w = csv.DictWriter(f, fieldnames=['source', 'name', 'coverage', 'iou'])
            w.writeheader()
            w.writerows(rows)
    return {'n_images': len(cov), 'coverage_mean': float(np.mean(cov)), 'coverage_std': float(np.std(cov)),
            'iou_mean': float(np.mean(ious)) if ious else None, 'iou_std': float(np.std(ious)) if ious else None,
            'n_iou': len(ious), 'n_empty': n_empty}


# ============================================================================
# TRAINER WITH MIXED PRECISION
# ============================================================================

class MFSSNetTrainer:
    """Trainer for MFSSNet model with Mixed Precision Training"""
    def __init__(self, model, train_loader, val_loader, device, config):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        self.config = config

        self.metrics_evaluator = MetricsEvaluator(
            metrics=config.metrics if hasattr(config, 'metrics') else ['psnr', 'ssim'],
            device=device
        )

        # Optimizer
        self.optimizer = optim.AdamW(
            model.parameters(),
            lr=config.lr,
            betas=(0.9, 0.999),
            weight_decay=config.weight_decay
        )

        # SCHEDULER: linear warm-up, then cosine annealing with warm restarts, updated
        # at every iteration.
        self.warmup_epochs = config.warmup_epochs if hasattr(config, 'warmup_epochs') else 5
        self.warmup_steps = self.warmup_epochs * len(train_loader)
        self.current_step = 0

        # Main scheduler: Cosine Annealing with Warm Restarts (cycles automatically)
        self.cosine_scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            self.optimizer,
            T_0=config.scheduler_t0 if hasattr(config, 'scheduler_t0') else 50,  # First restart at 50 epochs
            T_mult=config.scheduler_tmult if hasattr(config, 'scheduler_tmult') else 2,  # Double period after each restart
            eta_min=config.lr * 0.01  # Minimum LR is 1% of initial LR
        )
        # The warm-up starts from a learning rate of zero (the cosine schedule keeps
        # config.lr as its base value)
        if self.warmup_steps > 0:
            for group in self.optimizer.param_groups:
                group['lr'] = 0.0

        # Model selection uses the validation PSNR, which does not depend on the
        # loss weights (unlike the validation loss)
        self.selection_metric = getattr(config, 'selection_metric', 'psnr')

        # MIXED PRECISION: Initialize GradScaler
        self.use_amp = config.use_amp if hasattr(config, 'use_amp') else True
        self.scaler = GradScaler(enabled=self.use_amp)

        # Gradient clipping norm
        self.grad_clip_norm = config.grad_clip_norm if hasattr(config, 'grad_clip_norm') else 1.0

        self.criterion = MFSSNetLoss(
            alpha_pixel=config.alpha_pixel,
            alpha_perceptual=config.alpha_perceptual,
            alpha_edge=config.alpha_edge,
            alpha_char=config.alpha_char,
            alpha_stroke=config.alpha_stroke,
            alpha_style=getattr(config, 'alpha_style', 120.0),
            alpha_seg=getattr(config, 'alpha_seg', 0.5),
            alpha_bnd=getattr(config, 'alpha_bnd', 0.2),
            beta_kl=getattr(config, 'beta_kl', 1e-3)
        ).to(self.device)

        self.current_epoch = 0
        self.best_val_loss = float('inf')
        self.best_val_score = float('-inf')
        self.train_losses = defaultdict(list)
        self.val_losses = defaultdict(list)
        self.train_metrics = defaultdict(list)
        self.val_metrics = defaultdict(list)

        self.checkpoint_dir = Path(config.checkpoint_dir)
        self.results_dir = Path(config.results_dir)

        config.create_directories()

        logger.info(f"Mixed Precision Training: {'ENABLED' if self.use_amp else 'DISABLED'}")
        logger.info(f"Gradient Clipping Norm: {self.grad_clip_norm}")
        logger.info(f"Scheduler: Warmup({self.warmup_epochs}E) + CosineAnnealingWarmRestarts")

    def get_lr(self):
        """Get current learning rate with warmup"""
        if self.current_step < self.warmup_steps:
            # Linear warmup
            return self.config.lr * (self.current_step / self.warmup_steps)
        else:
            # Use cosine scheduler LR
            return self.optimizer.param_groups[0]['lr']

    def advance_schedule(self):
        """Learning rate of the next iteration: linear warm-up, then cosine
        annealing with warm restarts evaluated at the number of epochs elapsed
        since the start of training."""
        self.current_step += 1
        self.update_lr()
        if self.current_step >= self.warmup_steps:
            self.cosine_scheduler.step(self.current_step / len(self.train_loader))

    def update_lr(self):
        """Update learning rate with warmup logic"""
        if self.current_step < self.warmup_steps:
            # Warmup phase: linear increase
            lr = self.config.lr * (self.current_step / self.warmup_steps)
            for param_group in self.optimizer.param_groups:
                param_group['lr'] = lr
        # After warmup, cosine scheduler handles LR updates

    def save_checkpoint(self, is_best=False):
        checkpoint = {
            'epoch': self.current_epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'cosine_scheduler_state_dict': self.cosine_scheduler.state_dict(),
            'scaler_state_dict': self.scaler.state_dict(),  # Save scaler state
            'best_val_loss': self.best_val_loss,
            'best_val_score': self.best_val_score,
            'selection_metric': self.selection_metric,
            'current_step': self.current_step,
            'train_losses': dict(self.train_losses),
            'val_losses': dict(self.val_losses),
            'model_architecture': 'MFSSNet (CA-SSM + CG-CSWA + DSH + RWSE)',
            'model_config': model_kwargs(self.config),   # architecture
            'variant': 'full',
            'train_on': getattr(self.config, 'train_on', None),
            'seed': getattr(self.config, 'seed', None),
            'use_amp': self.use_amp,
            'grad_clip_norm': self.grad_clip_norm,
            'scheduler_type': 'Warmup + CosineAnnealingWarmRestarts'
        }

        # Atomic writes: the file is written under a temporary name and renamed only
        # once complete, so an interrupted save never leaves a truncated latest.pth
        def atomic_save(path):
            tmp = path.with_name(path.name + '.tmp')
            torch.save(checkpoint, tmp)
            os.replace(tmp, path)

        atomic_save(self.checkpoint_dir / 'latest.pth')
        # The loss curves are also kept as a small text file beside the checkpoints, so that the
        # validation curves can be rebuilt without the weights.
        try:
            hist_path = self.checkpoint_dir / 'history.json'
            prev = {}
            if hist_path.exists():
                try:
                    prev = json.loads(hist_path.read_text(encoding='utf-8')).get('epochs', {})
                except (ValueError, OSError):
                    prev = {}
            for part, losses in (('train', self.train_losses), ('val', self.val_losses)):
                for k, vals in dict(losses).items():
                    for j, x in enumerate(vals):
                        prev.setdefault(str(self.current_epoch - len(vals) + 1 + j), {}).setdefault(part, {})[k] = float(x)
            hist_path.write_text(json.dumps({'epoch': self.current_epoch, 'best_val_score': float(self.best_val_score),
                                             'selection_metric': self.selection_metric, 'epochs': prev}), encoding='utf-8')
        except Exception as e:                                           # the history file never stops a training run
            logger.warning(f"history.json not written ({type(e).__name__}: {e})")

        if is_best:
            atomic_save(self.checkpoint_dir / 'best.pth')
            logger.info(f"Saved best model at epoch {self.current_epoch}")

        if self.current_epoch % 25 == 0:
            atomic_save(self.checkpoint_dir / f'epoch_{self.current_epoch}.pth')

    def load_checkpoint(self, path=None):
        if path is None:
            path = self.checkpoint_dir / 'latest.pth'

        if not Path(path).exists():
            logger.info(f"No checkpoint at {path}")
            return

        checkpoint = torch.load(path, map_location=self.device, weights_only=False)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

        # Load scheduler states
        if 'cosine_scheduler_state_dict' in checkpoint:
            self.cosine_scheduler.load_state_dict(checkpoint['cosine_scheduler_state_dict'])

        # Load scaler state if available
        if 'scaler_state_dict' in checkpoint:
            self.scaler.load_state_dict(checkpoint['scaler_state_dict'])

        self.current_epoch = checkpoint.get('epoch', 0)
        self.current_step = checkpoint.get('current_step', 0)
        self.best_val_loss = checkpoint.get('best_val_loss', float('inf'))
        self.best_val_score = checkpoint.get('best_val_score', float('-inf'))
        # the loss histories continue across a resume
        for name in ('train_losses', 'val_losses'):
            hist = getattr(self, name)
            for k, vals in (checkpoint.get(name) or {}).items():
                hist[k] = list(vals)

        logger.info(f"Resumed MFSSNet training from epoch {self.current_epoch}")

    def train_epoch(self):
        self.model.train()
        epoch_losses = defaultdict(float)
        epoch_metrics = defaultdict(list)

        pbar = tqdm(self.train_loader, desc=f'MFSSNet Epoch {self.current_epoch+1}', leave=False)

        for batch in pbar:
            corrupted = batch['corrupted'].to(self.device)
            clean = batch['clean'].to(self.device)
            damage = batch['damage'].to(self.device)

            self.optimizer.zero_grad()

            # MIXED PRECISION: Forward pass with autocast
            with autocast('cuda', enabled=self.use_amp):
                pred, aux = self.model(corrupted, return_aux=True)
                total_loss, loss_dict = self.criterion(pred, clean, aux=aux, damage_target=damage)

            # MIXED PRECISION: Backward pass with scaler
            self.scaler.scale(total_loss).backward()

            # GRADIENT CLIPPING: Unscale gradients before clipping
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip_norm)

            # MIXED PRECISION: Optimizer step with scaler
            self.scaler.step(self.optimizer)
            self.scaler.update()

            # UPDATE LEARNING RATE (for the next iteration)
            self.advance_schedule()

            # Compute metrics (without autocast for accuracy)
            with torch.no_grad():
                metrics = self.metrics_evaluator.compute(pred, clean)
                for k, v in metrics.items():
                    epoch_metrics[k].append(v)

            epoch_losses['total'] += total_loss.item()
            for k, v in loss_dict.items():
                epoch_losses[k] += v

            current_lr = self.get_lr()
            pbar.set_postfix({
                'Loss': f"{total_loss.item():.4f}",
                'LR': f"{current_lr:.6f}",
                'Scale': f"{self.scaler.get_scale():.0f}",
                'Phase': 'Warmup' if self.current_step < self.warmup_steps else 'Cosine'
            })

        num_batches = len(self.train_loader)
        for k in epoch_losses:
            epoch_losses[k] /= num_batches
            self.train_losses[k].append(epoch_losses[k])

        for k, v in epoch_metrics.items():
            if v:
                avg = sum(v) / len(v)
                self.train_metrics[k].append(avg)

        return epoch_losses, {k: sum(v)/len(v) for k, v in epoch_metrics.items() if v}

    def validate_epoch(self):
        self.model.eval()
        epoch_losses = defaultdict(float)
        epoch_metrics = defaultdict(list)

        with torch.no_grad():
            pbar = tqdm(self.val_loader, desc='MFSSNet Validation', leave=False)

            for batch in pbar:
                corrupted = batch['corrupted'].to(self.device)
                clean = batch['clean'].to(self.device)
                damage = batch['damage'].to(self.device)

                # Use autocast for validation too (eval mode: z = mu, deterministic)
                with autocast('cuda', enabled=self.use_amp):
                    pred, aux = self.model(corrupted, return_aux=True)
                    total_loss, loss_dict = self.criterion(pred, clean, aux=aux, damage_target=damage)

                metrics = self.metrics_evaluator.compute(pred, clean)
                for k, v in metrics.items():
                    epoch_metrics[k].append(v)

                epoch_losses['total'] += total_loss.item()
                for k, v in loss_dict.items():
                    epoch_losses[k] += v

        num_batches = len(self.val_loader)
        for k in epoch_losses:
            epoch_losses[k] /= num_batches
            self.val_losses[k].append(epoch_losses[k])

        for k, v in epoch_metrics.items():
            if v:
                avg = sum(v) / len(v)
                self.val_metrics[k].append(avg)

        return epoch_losses, {k: sum(v)/len(v) for k, v in epoch_metrics.items() if v}

    def train(self, epochs):
        logger.info(f"Starting MFSSNet training for {epochs} epochs")
        logger.info(f"Model: MFSSNet with {sum(p.numel() for p in self.model.parameters()):,} parameters")
        logger.info("Novel Components: CA-SSM + CG-CSWA + DSH + RWSE")
        logger.info(f"Mixed Precision: {'ENABLED' if self.use_amp else 'DISABLED'}")
        logger.info(f"Gradient Clipping: {self.grad_clip_norm}")
        logger.info(f"Scheduler: Warmup({self.warmup_epochs}E) + CosineAnnealingWarmRestarts")

        for epoch in range(self.current_epoch, epochs):
            train_losses, train_metrics = self.train_epoch()
            val_losses, val_metrics = self.validate_epoch()

            # Selection criterion (validation PSNR)
            if self.selection_metric not in val_metrics:
                raise RuntimeError(
                    f"validation metric {self.selection_metric!r} could not be computed on any batch "
                    f"(got {sorted(val_metrics)}); see the 'Error computing' warnings above")
            score = val_metrics[self.selection_metric]

            current_lr = self.get_lr()
            log_msg = (f"Epoch {epoch+1}/{epochs} - "
                      f"LR: {current_lr:.6f} - "
                      f"Train Loss: {train_losses['total']:.4f} - "
                      f"Val Loss: {val_losses['total']:.4f}")

            for k in ['psnr', 'ssim']:
                if k in train_metrics and k in val_metrics:
                    log_msg += f" - Train {k.upper()}: {train_metrics[k]:.3f}"
                    log_msg += f" - Val {k.upper()}: {val_metrics[k]:.3f}"

            logger.info(log_msg)

            self.best_val_loss = min(self.best_val_loss, val_losses['total'])
            is_best = score > self.best_val_score
            if is_best:
                self.best_val_score = score
                logger.info(f"New best validation {self.selection_metric.upper()}: {score:.4f}")

            self.current_epoch = epoch + 1
            self.save_checkpoint(is_best)


# ============================================================================
# INFERENCER
# ============================================================================

class MFSSNetInferencer:
    """Inferencer for MFSSNet model with gradient attribution maps and multiple analysis tools"""
    def __init__(self, model, device, config, checkpoint_path=None):
        self.device = device
        self.model = model.to(device)
        self.config = config
        self.metrics_evaluator = MetricsEvaluator(
            metrics=config.metrics if hasattr(config, 'metrics') else ['psnr', 'ssim'],
            device=device
        )

        if checkpoint_path:
            self.load_checkpoint(checkpoint_path)

        self.model.eval()

    def load_checkpoint(self, path):
        checkpoint = torch.load(path, map_location=self.device, weights_only=False)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        logger.info(f"Loaded MFSSNet model from {path}")

    def compute_lam(self, corrupted, restored, clean):
        """Gradient attribution map: absolute gradient of the squared error between the
        output and the intact image with respect to the input pixels, averaged over the
        channels and normalized to [0, 1] (Fig. 5(k)). This is a plain gradient saliency
        map, not the path-integrated local attribution map (LAM) of Gu and Dong."""
        corrupted_input = corrupted.clone().detach().requires_grad_(True)

        # We need to forward pass again with grad enabled
        output = self.model(corrupted_input)

        # Loss for backprop (MSE with ground truth)
        loss = F.mse_loss(output, clean)

        self.model.zero_grad()
        loss.backward()

        lam = torch.abs(corrupted_input.grad)
        lam = lam.mean(dim=1, keepdim=True)
        lam = (lam - lam.min()) / (lam.max() - lam.min() + 1e-8)

        return lam.detach()

    def visualize_spectral_density(self, image):
        """Log-magnitude spectrum of the centred 2-D Fourier transform of the grey-level
        image (Fig. 5(h)-(j))."""
        img_np = image.mean(dim=0).cpu().numpy()
        f = np.fft.fft2(img_np)
        fshift = np.fft.fftshift(f)
        magnitude_spectrum = 20 * np.log(np.abs(fshift) + 1e-8)
        return magnitude_spectrum

    def visualize_pixel_distribution(self, ax, image, label, color):
        """Visualize histogram of pixel intensities"""
        img_np = image.cpu().numpy().flatten()
        ax.hist(img_np, bins=50, color=color, alpha=0.5, label=label, density=True)

    def _edges(self, restored, clean):
        sobel_x = torch.tensor([[[[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]]]).float().to(self.device)
        sobel_y = torch.tensor([[[[-1, -2, -1], [0, 0, 0], [1, 2, 1]]]]).float().to(self.device)

        restored_gray = restored.mean(dim=0, keepdim=True).unsqueeze(0)
        clean_gray = clean.mean(dim=0, keepdim=True).unsqueeze(0)

        restored_edges = torch.sqrt(
            F.conv2d(restored_gray, sobel_x, padding=1)**2 +
            F.conv2d(restored_gray, sobel_y, padding=1)**2
        )
        clean_edges = torch.sqrt(
            F.conv2d(clean_gray, sobel_x, padding=1)**2 +
            F.conv2d(clean_gray, sobel_y, padding=1)**2
        )
        return restored_edges, clean_edges

    def save_individual_components(self, corrupted, restored, clean, image_id, save_dir):
        """Save individual image components and analysis maps"""
        individual_dir = Path(save_dir) / 'individual_components' / f'{image_id}'
        individual_dir.mkdir(parents=True, exist_ok=True)

        to_pil = transforms.ToPILImage()

        # 1. Save basic images
        to_pil(torch.clamp(corrupted, 0.0, 1.0)).save(individual_dir / f'{image_id}_01_corrupted.png')
        to_pil(torch.clamp(restored, 0.0, 1.0)).save(individual_dir / f'{image_id}_02_restored.png')
        to_pil(torch.clamp(clean, 0.0, 1.0)).save(individual_dir / f'{image_id}_03_ground_truth.png')

        # 2. Save difference map
        diff_np = torch.abs(restored - clean).mean(dim=0).cpu().numpy()
        diff_colored = plt.cm.hot(diff_np / (diff_np.max() + 1e-8))[:, :, :3]
        Image.fromarray((diff_colored * 255).astype(np.uint8)).save(individual_dir / f'{image_id}_04_difference_map.png')

        # 3. Save attribution map
        with torch.enable_grad():
            lam = self.compute_lam(corrupted.unsqueeze(0), restored.unsqueeze(0), clean.unsqueeze(0))

        lam_np = lam.squeeze().cpu().numpy()
        lam_colored = plt.cm.jet(lam_np)[:, :, :3]
        Image.fromarray((lam_colored * 255).astype(np.uint8)).save(individual_dir / f'{image_id}_05_lam.png')

        # 4. Save attribution overlay
        corrupted_np = torch.clamp(corrupted, 0.0, 1.0).cpu().permute(1, 2, 0).numpy()
        overlay = np.clip(0.6 * corrupted_np + 0.4 * lam_colored, 0.0, 1.0)
        Image.fromarray((overlay * 255).astype(np.uint8)).save(individual_dir / f'{image_id}_06_lam_overlay.png')

        # 5. Save high attribution regions (binary mask)
        lam_binary = (lam_np > lam_np.mean()).astype(np.uint8) * 255
        Image.fromarray(lam_binary, mode='L').save(individual_dir / f'{image_id}_07_high_attribution.png')

        # 6. Save edge maps
        restored_edges, clean_edges = self._edges(restored, clean)

        restored_edges_np = restored_edges.squeeze().cpu().numpy()
        restored_edges_np = (restored_edges_np / (restored_edges_np.max() + 1e-8) * 255).astype(np.uint8)
        Image.fromarray(restored_edges_np, mode='L').save(individual_dir / f'{image_id}_08_restored_edges.png')

        clean_edges_np = clean_edges.squeeze().cpu().numpy()
        clean_edges_np = (clean_edges_np / (clean_edges_np.max() + 1e-8) * 255).astype(np.uint8)
        Image.fromarray(clean_edges_np, mode='L').save(individual_dir / f'{image_id}_09_ground_truth_edges.png')

        # 7. Save edge difference map
        edge_diff_np = torch.abs(restored_edges - clean_edges).squeeze().cpu().numpy()
        edge_diff_colored = plt.cm.hot(edge_diff_np / (edge_diff_np.max() + 1e-8))[:, :, :3]
        Image.fromarray((edge_diff_colored * 255).astype(np.uint8)).save(individual_dir / f'{image_id}_10_edge_difference.png')

        logger.info(f"Saved individual components to {individual_dir}")

    def visualize_lam(self, corrupted, restored, clean, save_path):
        """Visualize the gradient attribution map"""
        with torch.enable_grad():
            lam = self.compute_lam(corrupted.unsqueeze(0), restored.unsqueeze(0), clean.unsqueeze(0))

        corrupted_np = torch.clamp(corrupted, 0.0, 1.0).cpu().permute(1, 2, 0).numpy()
        lam_np = lam.squeeze().cpu().numpy()

        fig, axes = plt.subplots(1, 3, figsize=(18, 6))

        axes[0].imshow(corrupted_np)
        axes[0].set_title('Corrupted Input', fontsize=16, fontweight='bold')
        axes[0].axis('off')

        im1 = axes[1].imshow(lam_np, cmap='jet', interpolation='bilinear')
        axes[1].set_title('Gradient Attribution Map', fontsize=16, fontweight='bold')
        axes[1].axis('off')
        plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

        heatmap = plt.cm.jet(lam_np)[:, :, :3]
        overlay = np.clip(0.6 * corrupted_np + 0.4 * heatmap, 0.0, 1.0)
        axes[2].imshow(overlay)
        axes[2].set_title('Attribution Overlay', fontsize=16, fontweight='bold')
        axes[2].axis('off')

        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        plt.close()

    def create_comprehensive_analysis(self, corrupted, restored, clean, image_id, save_dir):
        """Comprehensive analysis with attribution map, FFT, histograms and edge maps in one file."""
        to_pil = transforms.ToPILImage()
        corrupted_img = to_pil(torch.clamp(corrupted, 0.0, 1.0))
        restored_img = to_pil(torch.clamp(restored, 0.0, 1.0))
        clean_img = to_pil(torch.clamp(clean, 0.0, 1.0))

        with torch.no_grad():
            metrics = self.metrics_evaluator.compute(restored.unsqueeze(0), clean.unsqueeze(0))

        with torch.enable_grad():
            lam = self.compute_lam(corrupted.unsqueeze(0), restored.unsqueeze(0), clean.unsqueeze(0))
        lam_np = lam.squeeze().cpu().numpy()

        # Grid Setup: 4 Rows x 4 Columns
        fig = plt.figure(figsize=(24, 20))
        gs = fig.add_gridspec(4, 4, hspace=0.35, wspace=0.35)

        # --- Row 1: Visual Comparison ---
        for col, (img, title) in enumerate(((corrupted_img, 'Corrupted'), (restored_img, 'Restored'),
                                            (clean_img, 'Ground Truth'))):
            ax = fig.add_subplot(gs[0, col])
            ax.imshow(img)
            ax.set_title(title, fontsize=14, fontweight='bold')
            ax.axis('off')

        diff = torch.abs(restored - clean).mean(dim=0)
        ax4 = fig.add_subplot(gs[0, 3])
        im4 = ax4.imshow(diff.cpu().numpy(), cmap='hot', interpolation='bilinear')
        ax4.set_title('Error Heatmap', fontsize=14, fontweight='bold')
        ax4.axis('off')
        plt.colorbar(im4, ax=ax4, fraction=0.046, pad=0.04)

        # --- Row 2: Attribution map ---
        ax5 = fig.add_subplot(gs[1, 0])
        im5 = ax5.imshow(lam_np, cmap='jet', interpolation='bilinear')
        ax5.set_title('Gradient Attribution Map', fontsize=14, fontweight='bold')
        ax5.axis('off')
        plt.colorbar(im5, ax=ax5, fraction=0.046, pad=0.04)

        corrupted_np = torch.clamp(corrupted, 0.0, 1.0).cpu().permute(1, 2, 0).numpy()
        overlay = np.clip(0.6 * corrupted_np + 0.4 * plt.cm.jet(lam_np)[:, :, :3], 0.0, 1.0)

        ax6 = fig.add_subplot(gs[1, 1])
        ax6.imshow(overlay)
        ax6.set_title('Attribution on Input', fontsize=14, fontweight='bold')
        ax6.axis('off')

        # Metrics Display
        ax7 = fig.add_subplot(gs[1, 2:])
        metric_text = '\n'.join([f"{k.upper()}: {v:.4f}" for k, v in metrics.items()])
        ax7.text(0.5, 0.5, metric_text, ha='center', va='center', fontsize=18,
                 bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8, pad=1.0),
                 family='monospace')
        ax7.set_title('Quality Metrics', fontsize=14, fontweight='bold')
        ax7.axis('off')

        # --- Row 3: Frequency Analysis (FFT) ---
        for col, (img, title) in enumerate(((corrupted, 'FFT Corrupted'), (restored, 'FFT Restored'),
                                            (clean, 'FFT Ground Truth'))):
            ax = fig.add_subplot(gs[2, col])
            ax.imshow(self.visualize_spectral_density(img), cmap='viridis')
            ax.set_title(title, fontsize=14, fontweight='bold')
            ax.axis('off')

        # Histogram
        ax11 = fig.add_subplot(gs[2, 3])
        self.visualize_pixel_distribution(ax11, restored, 'Restored', 'blue')
        self.visualize_pixel_distribution(ax11, clean, 'GT', 'green')
        ax11.set_title('Pixel Histogram', fontsize=14, fontweight='bold')
        ax11.legend()

        # --- Row 4: Edge Maps ---
        restored_edges, clean_edges = self._edges(restored, clean)

        ax12 = fig.add_subplot(gs[3, 0])
        ax12.imshow(restored_edges.squeeze().cpu().numpy(), cmap='gray')
        ax12.set_title('Restored Edges', fontsize=14, fontweight='bold')
        ax12.axis('off')

        ax13 = fig.add_subplot(gs[3, 1])
        ax13.imshow(clean_edges.squeeze().cpu().numpy(), cmap='gray')
        ax13.set_title('Ground Truth Edges', fontsize=14, fontweight='bold')
        ax13.axis('off')

        ax14 = fig.add_subplot(gs[3, 2])
        ax14.imshow(torch.abs(restored_edges - clean_edges).squeeze().cpu().numpy(), cmap='hot')
        ax14.set_title('Edge Difference', fontsize=14, fontweight='bold')
        ax14.axis('off')

        plt.suptitle(f'MFSSNet Analysis - {image_id}', fontsize=22, fontweight='bold', y=0.98)

        save_path = Path(save_dir) / 'comprehensive' / f'analysis_{image_id}.png'
        plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
        plt.close()

        return metrics

    def infer_image_set(self, image_paths, save_dir=None):
        """Infer a set of (corrupted, ground truth) image pairs, with the full analysis of each."""
        save_dir = Path(self.config.analysis_dir) / 'custom_inference' if save_dir is None else Path(save_dir)

        save_dir.mkdir(parents=True, exist_ok=True)
        (save_dir / 'comprehensive').mkdir(exist_ok=True)
        (save_dir / 'lam_maps').mkdir(exist_ok=True)
        (save_dir / 'individual_components').mkdir(exist_ok=True)

        transform = transforms.Compose([transforms.ToTensor()])

        all_results = []

        # The main loop is no_grad; gradients are enabled locally for the attribution maps.
        with torch.no_grad():
            for idx, (corrupted_path, clean_path) in enumerate(tqdm(image_paths, desc='Processing images')):
                try:
                    corrupted = transform(Image.open(corrupted_path).convert('RGB')).unsqueeze(0).to(self.device)
                    clean = transform(Image.open(clean_path).convert('RGB')).unsqueeze(0).to(self.device)

                    restored = torch.clamp(self.model(corrupted), 0.0, 1.0)

                    image_id = f'{idx:03d}_{Path(corrupted_path).stem}'

                    metrics = self.create_comprehensive_analysis(
                        corrupted.squeeze(0), restored.squeeze(0), clean.squeeze(0), image_id, save_dir)

                    self.visualize_lam(corrupted.squeeze(0), restored.squeeze(0), clean.squeeze(0),
                                       save_dir / 'lam_maps' / f'lam_{image_id}.png')

                    self.save_individual_components(
                        corrupted.squeeze(0), restored.squeeze(0), clean.squeeze(0), image_id, save_dir)

                    all_results.append({
                        'index': idx,
                        'image_id': image_id,
                        'corrupted_path': str(corrupted_path),
                        'clean_path': str(clean_path),
                        'metrics': metrics,
                        'components_dir': str(save_dir / 'individual_components' / image_id)
                    })

                    logger.info(f"Processed {idx+1}/{len(image_paths)}: {image_id}")
                    logger.info("  Metrics: " + ", ".join([f"{k}: {v:.4f}" for k, v in metrics.items()]))

                except Exception as e:
                    logger.error(f"Error processing {corrupted_path}: {str(e)}")
                    continue

        if all_results:
            logger.info("\n" + "="*80)
            logger.info("MFSSNet Custom Image Set Inference Summary")
            logger.info("="*80)
            logger.info(f"Total images processed: {len(all_results)}")

            all_metrics = defaultdict(list)
            for result in all_results:
                for k, v in result['metrics'].items():
                    all_metrics[k].append(v)

            final_metrics = {}
            for k, values in all_metrics.items():
                final_metrics[k] = {'mean': np.mean(values), 'std': np.std(values),
                                    'min': np.min(values), 'max': np.max(values)}
                logger.info(f"{k.upper():>12}: {np.mean(values):>7.4f} ± {np.std(values):.4f}")
            logger.info("="*80)

            summary = {'final_metrics': final_metrics, 'num_samples': len(all_results),
                       'results': all_results, 'model': 'MFSSNet'}
            with open(save_dir / 'custom_inference_summary.json', 'w') as f:
                json.dump(summary, f, indent=2, default=str)

        return all_results

    def batch_inference(self, data_loader, num_samples=100):
        """Batch inference using a DataLoader, with the full analysis of every sample processed."""
        self.config.create_directories()

        output_dir = Path(self.config.analysis_dir)
        all_metrics = defaultdict(list)
        count = 0
        to_pil = transforms.ToPILImage()
        manipulated_dir = Path(self.config.manipulated_dir)

        with torch.no_grad():
            pbar = tqdm(data_loader, desc='MFSSNet Inference')

            for batch in pbar:
                if num_samples is not None and count >= num_samples:
                    break

                corrupted = batch['corrupted'].to(self.device)
                clean = batch['clean'].to(self.device)
                image_ids = batch['image_id']

                restored = torch.clamp(self.model(corrupted), 0.0, 1.0)

                for i in range(min(corrupted.size(0), num_samples - count if num_samples else corrupted.size(0))):
                    metrics = self.create_comprehensive_analysis(
                        corrupted[i], restored[i], clean[i], image_ids[i], output_dir)

                    self.visualize_lam(corrupted[i], restored[i], clean[i],
                                       output_dir / 'lam_maps' / f'lam_{image_ids[i]}.png')

                    self.save_individual_components(corrupted[i], restored[i], clean[i], image_ids[i], output_dir)

                    to_pil(torch.clamp(corrupted[i].cpu(), 0.0, 1.0)).save(manipulated_dir / 'corrupted' / f"{image_ids[i]}.png")
                    to_pil(torch.clamp(restored[i].cpu(), 0.0, 1.0)).save(manipulated_dir / 'restored' / f"{image_ids[i]}.png")
                    to_pil(torch.clamp(clean[i].cpu(), 0.0, 1.0)).save(manipulated_dir / 'clean' / f"{image_ids[i]}.png")

                    for k, v in metrics.items():
                        all_metrics[k].append(v)

                    count += 1

                if all_metrics:
                    pbar.set_postfix({k: f"{sum(v)/len(v):.3f}" for k, v in all_metrics.items()})

        final_metrics = {k: {'mean': sum(v) / len(v), 'std': np.std(v), 'min': min(v), 'max': max(v)}
                         for k, v in all_metrics.items() if v}

        logger.info("=== MFSSNet Evaluation Results ===")
        for k, stats in final_metrics.items():
            logger.info(f"{k.upper()}: {stats['mean']:.4f} ± {stats['std']:.4f}")

        with open(output_dir / 'evaluation_summary.json', 'w') as f:
            json.dump({'final_metrics': final_metrics, 'num_samples': count, 'model': 'MFSSNet'},
                      f, indent=2, default=str)

        return final_metrics


# ============================================================================
# PROFILING HELPER
# ============================================================================

def profile_model_and_components(model, config, device):
    print("\n" + "="*85)
    print("MODEL PROFILING: Parameters, Size, GFLOPS, Inference Speed")
    print("="*85)

    try:
        from thop import profile as thop_profile
        has_thop = True
    except ImportError:
        has_thop = False
        print("WARNING: 'thop' library not found. FLOPs calculation skipped.")
        print("Run 'pip install thop' to enable GFLOPS calculations.\n")

    def run_profiling(name, module, dummy_inputs):
        if module is None:
            return

        module.eval()
        module.to(device)

        if isinstance(dummy_inputs, torch.Tensor):
            dummy_inputs = (dummy_inputs,)
        dummy_inputs = tuple(t.to(device) if isinstance(t, torch.Tensor) else t for t in dummy_inputs)

        # 1. Parameters & Size
        params = sum(p.numel() for p in module.parameters())
        size_mb = params * 4 / (1024 ** 2)

        # 2. FLOPs
        flops_str = "N/A"
        if has_thop:
            try:
                macs, _ = thop_profile(module, inputs=dummy_inputs, verbose=False)
                flops_str = f"{(macs * 2) / (10 ** 9):.3f} GFLOPS"
            except Exception:
                flops_str = "Error/Unsupported"

        # 3. Inference Speed (Pure)
        warmup, iters, avg_time = 10, 50, 0
        try:
            with torch.no_grad():
                for _ in range(warmup):
                    module(*dummy_inputs)
                if device.type == 'cuda':
                    torch.cuda.synchronize()
                start_time = time.perf_counter()
                for _ in range(iters):
                    module(*dummy_inputs)
                if device.type == 'cuda':
                    torch.cuda.synchronize()
                avg_time = (time.perf_counter() - start_time) / iters
        except Exception:
            pass

        time_str = f"{avg_time*1000:.2f} ms" if avg_time > 0 else "Error"
        print(f"{name:<25} | Params: {params/1e6:>7.3f}M | Size: {size_mb:>7.2f}MB | {flops_str:>15} | Speed: {time_str:>10} / pass")

    B, C, H, W = 1, config.in_channels, config.img_size[0], config.img_size[1]
    dummy_img = torch.randn(B, C, H, W)

    run_profiling("Full MFSSNet", model, (dummy_img,))
    print("-" * 85)
    print("Component Profiling:")

    run_profiling("1. SFE", model.sfe, (dummy_img,))

    hfe_in = torch.randn(B, config.dim, H, W)
    damage_in = torch.zeros(B, 1, H, W)
    if len(model.hfe_blocks) > 0 and len(model.hfe_blocks[0]) > 0:
        run_profiling("2. HFE Block (Stage 1)", model.hfe_blocks[0][0], (hfe_in,))

    n_down = len(config.num_blocks) - 1
    bn_dim = config.dim * (2 ** n_down)
    bn_h, bn_w = H // (2 ** n_down), W // (2 ** n_down)
    bn_in = torch.randn(B, bn_dim, bn_h, bn_w)
    z_dummy = torch.randn(B, config.style_dim)

    if len(model.bne) > 0:
        run_profiling("3. BNE Block", model.bne[0], (bn_in, z_dummy))

    hfr_in = torch.randn(B, bn_dim // 2, bn_h * 2, bn_w * 2)
    if len(model.hfr_blocks) > 0 and len(model.hfr_blocks[0]) > 0:
        run_profiling("4. HFR Block (Stage 1)", model.hfr_blocks[0][0], (hfr_in, z_dummy))

    run_profiling("5. DSH", model.dsh, (hfe_in,))
    run_profiling("6. RWSE", model.rwse, (hfe_in, damage_in))

    patch_dim = config.patch_size * config.patch_size * config.dim
    seq_len = (H // config.patch_size) * (W // config.patch_size)
    run_profiling("7. PRT", model.prt, (torch.randn(B, seq_len, patch_dim),))

    run_profiling("8. DFR", model.dfr, (hfe_in,))

    print("-" * 85)
    print("Inner Sub-components (based on Stage 1 shapes):")
    if len(model.hfe_blocks) > 0 and len(model.hfe_blocks[0]) > 0:
        hrb = model.hfe_blocks[0][0]
        run_profiling("- CA-SSM", hrb.srb.ca_ssm, (hfe_in,))
        run_profiling("- CG-CSWA", hrb.srb.cg_cswa, (hfe_in,))
        run_profiling("- GGCF", hrb.srb.ggcf, (hfe_in,))
        run_profiling("- FRB Full", hrb.frb, (hfe_in,))
        run_profiling("  - AFM", hrb.frb.afm, (hfe_in,))
        run_profiling("  - FDS", hrb.frb.fds, (hfe_in,))
        run_profiling("  - FFM", hrb.frb.ffm, (hfe_in, hfe_in))

    print("="*85 + "\n")


# ============================================================================
# MAIN
# ============================================================================

def main():
    """Main training and evaluation function for MFSSNet"""
    config = Config()   # all the settings of the paper are those of Config (Section 4.1)
    config.resume = True

    # Benchmark protocol: one model per benchmark, trained on its official
    # training set only, selected on a validation split of that training set,
    # and evaluated once on the untouched official test set.
    config.train_on = 'st'  # run a second time with 'ht'
    config.checkpoint_dir = f'./runnings/{config.train_on}'
    set_seed(config.seed)

    config.create_directories()

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")
    print("Model: MFSSNet with CA-SSM + CG-CSWA + DSH + RWSE")
    print(f"Mixed Precision: {'ENABLED' if config.use_amp else 'DISABLED'}")
    print(f"Gradient Clipping: {config.grad_clip_norm}")
    print(f"Scheduler: Warmup({config.warmup_epochs}E) + CosineAnnealingWarmRestarts(T0={config.scheduler_t0}, Tmult={config.scheduler_tmult})")

    # Datasets: official training set (split into train / validation) and official test set
    train_dataset, val_dataset, test_dataset = build_benchmark_datasets(config)

    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False, num_workers=4, pin_memory=True)

    print(f"[{config.train_on}] Train: {len(train_dataset)}, Validation (from the training set): {len(val_dataset)}, "
          f"Test (official, never used for selection): {len(test_dataset)}")
    # Exact splits of this run (image lists and SHA-256 fingerprints), to be published
    Path(config.checkpoint_dir).mkdir(parents=True, exist_ok=True)
    with open(Path(config.checkpoint_dir) / f'split_{config.train_on}.json', 'w') as f:
        json.dump(split_manifest(train_dataset, val_dataset, test_dataset), f)

    model = MFSSNet(**model_kwargs(config))
    print(f"MFSSNet initialized with {sum(p.numel() for p in model.parameters()):,} parameters")

    profile_model_and_components(model, config, device)

    trainer = MFSSNetTrainer(model, train_loader, val_loader, device, config)

    if config.resume:
        trainer.load_checkpoint()

    trainer.train(config.epochs)

    print("Starting MFSSNet inference...")
    inferencer = MFSSNetInferencer(model, device, config, checkpoint_path=trainer.checkpoint_dir / 'best.pth')

    # Benchmark evaluation on the FULL official test set (metric code of the benchmark)
    test_dir = Path(config.results_dir) / f'test_{config.train_on}'
    final_metrics = evaluate_benchmark(model, test_dataset, test_dir, device)
    final_metrics['timing'] = time_inference(model, test_dataset, device)
    final_metrics['damage_head'] = evaluate_damage_head(model, test_dataset, device,
                                                        out_csv=test_dir / 'damage_head.csv')
    with open(test_dir / 'summary.json', 'w') as f:
        json.dump(final_metrics, f, indent=2)
    print(final_metrics)

    # Qualitative analysis (attribution maps, figures) on part of the test set; not used for the tables
    test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False, num_workers=4)
    inferencer.batch_inference(test_loader, num_samples=600)

    # Custom image pairs for the qualitative figures (corrupted, intact)
    st_ids = [f'{i:04d}' for i in list(range(0, 1)) + list(range(4, 28)) + list(range(62, 108))
              + [367, 537, 564, 565, 599, 733, 734]]
    st_pairs = [(f'/kaggle/working/stt_test/t_incomplete/{i}.png', f'/kaggle/working/stt_test/t_f/{i}.png')
                for i in st_ids]
    ht_ids = [f'{i:05d}' for i in list(range(0, 26)) + list(range(1000, 1010))]
    ht_pairs = [(f'/kaggle/working/htt_test/Corrupted_Image/{i}.png', f'/kaggle/working/htt_test/Intact_Image/{i}.png')
                for i in ht_ids]
    ht_pairs_2 = [(f'/kaggle/working/htt_test/Corrupted_Image/{i:05d}.png',
                   f'/kaggle/working/htt_test/Intact_Image/{i:05d}.png') for i in range(1509, 1521)]
    image_pairs = ht_pairs_2   # or st_pairs / ht_pairs

    results = inferencer.infer_image_set(image_paths=image_pairs, save_dir='htt_results')

    for result in results:
        print(f"Image {result['index']}: PSNR = {result['metrics']['psnr']:.2f}")


if __name__ == '__main__':
    main()
